# LatihIsyarat: pertahankan model 24 huruf, latih J dan Z

**Satu notebook, Run all. Aplikasi belum diubah.**

1. Model 24 huruf saat ini dibawa utuh dan **tidak dilatih ulang**.
2. Crop MediaPipe dan keputusan seperti aplikasi digunakan untuk evaluasi MNIST/Hugging Face.
3. Model GRU baru dilatih untuk **J, Z, dan OTHER** dari urutan landmark video.
4. Bukti lengkap disimpan ke Drive dan dikemas menjadi **satu ZIP otomatis diunduh**.

Pilih **Runtime > Change runtime type > T4 GPU**, lalu **Run all**.
Izinkan mount Drive. Unduhan pertama SigNN sekitar 4,17 GB, kemudian dipakai ulang.

Target 99,5% dilaporkan jujur setelah test. Akurasi sempurna tidak dijamin.
HF sudah pernah dipakai melatih model 24 huruf, sehingga hasilnya dinamai diagnostik.
OTHER default sintetis. Tambahkan rekaman gerakan lain di Drive untuk validasi yang lebih kuat.


In [ ]:
# Instalasi dependency. TensorFlow GPU bawaan Colab dipertahankan.
import importlib.metadata as metadata
import subprocess
import sys
from packaging.requirements import Requirement

DEPENDENCIES = ['mediapipe==0.10.32', 'opencv-contrib-python==4.11.0.86', 'kagglehub==0.3.13', 'numpy>=1.26,<2.3', 'pandas>=2.2,<3', 'pyarrow>=19,<24', 'scikit-learn>=1.6,<2', 'matplotlib>=3.9,<4', 'Pillow>=11,<13', 'requests>=2.32,<3']
# Hormati batas NumPy TensorFlow yang sudah terpasang, termasuk TF 2.19.
try:
    tf_requirements = metadata.requires('tensorflow') or []
except metadata.PackageNotFoundError as exc:
    raise RuntimeError('Gunakan runtime Google Colab dengan TensorFlow GPU.') from exc
for item in tf_requirements:
    requirement = Requirement(item)
    if requirement.name.lower() == 'numpy' and (requirement.marker is None or requirement.marker.evaluate()):
        DEPENDENCIES.append(str(requirement))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *DEPENDENCIES])
print('Dependency siap. Training memakai proses baru agar tidak perlu restart kernel.')


In [ ]:
# Kode dan model lama sudah tertanam di notebook ini, diverifikasi SHA-256.
import base64
import hashlib
import json
from pathlib import Path

SOURCE_DIR = Path('/content/latihisyarat_jz_source')
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_FILES = json.loads('{"train_all.py":{"base64":"IiIiT25lIGVudHJ5IHBvaW50OiBwcmVzZXJ2ZS9ldmFsdWF0ZSB0aGUgY3VycmVudCBBU0wyNCBtb2RlbCwgdHJhaW4gdGVtcG9yYWwgSi9aLgoKVGhpcyB3b3JrZmxvdyBjcmVhdGVzIG5ldyBhcnRpZmFjdHMgb25seS4gSXQgbmV2ZXIgbW9kaWZpZXMgdGhlIEZsdXR0ZXIgYXBwLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjb250ZXh0bGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc2h1dGlsCmltcG9ydCBzeXMKaW1wb3J0IHRyYWNlYmFjawppbXBvcnQgdXJsbGliLnJlcXVlc3QKaW1wb3J0IHppcGZpbGUKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKZnJvbSB0cmFpbmluZ19jb21tb24gaW1wb3J0IGpzb25fd3JpdGUsIHJlY29yZF9lbnZpcm9ubWVudCwgc2hhMjU2X2ZpbGUsIHN0YWJsZV9oYXNoLCB3cml0ZV9jc3YKCgpERUZBVUxUX0NPTkZJRyA9IHsKICAgICJzZWVkIjogNDIsCiAgICAic3BsaXRfc2VlZCI6IDIwMjYsCiAgICAic3RhdGljX21vZGUiOiAiZXZhbHVhdGVfZXhpc3RpbmciLAogICAgInZhbGlkYXRpb25fZnJhY3Rpb24iOiAwLjE1LAogICAgInRlc3RfZnJhY3Rpb24iOiAwLjE1LAogICAgImR5bmFtaWNfZXBvY2hzIjogNjAsCiAgICAiYmF0Y2hfc2l6ZSI6IDI1NiwKICAgICJsZWFybmluZ19yYXRlIjogMC4wMDEsCiAgICAic3RhdGljX3JlYWxfZW5hYmxlZCI6IFRydWUsCiAgICAic2VxdWVuY2VfbGVuZ3RoIjogNDgsCiAgICAiZHluYW1pY19zYW1wbGVfZnBzIjogMTUsCiAgICAibWluX2NvbmZpZGVuY2UiOiAwLjYwLAogICAgInRhcmdldF9keW5hbWljX2FjY3VyYWN5IjogMC45OTUsCiAgICAibWF4X21vZGVsX21pYiI6IDEwLAogICAgInJlcXVpcmVfZ3B1IjogVHJ1ZSwKICAgICJ1c2VfZHJpdmUiOiBUcnVlLAogICAgImRyaXZlX2RpciI6ICIvY29udGVudC9kcml2ZS9NeURyaXZlL0xhdGloSXN5YXJhdF9UcmFpbmluZ19KX1oiLAogICAgImF1dG9fZG93bmxvYWQiOiBUcnVlLAp9CgpIQU5EX1RBU0tfVVJMID0gKAogICAgImh0dHBzOi8vc3RvcmFnZS5nb29nbGVhcGlzLmNvbS9tZWRpYXBpcGUtbW9kZWxzL2hhbmRfbGFuZG1hcmtlci8iCiAgICAiaGFuZF9sYW5kbWFya2VyL2Zsb2F0MTYvMS9oYW5kX2xhbmRtYXJrZXIudGFzayIKKQoKCmNsYXNzIF9UZWU6CiAgICBkZWYgX19pbml0X18oc2VsZiwgb3JpZ2luYWwsIGZpbGUpOgogICAgICAgIHNlbGYub3JpZ2luYWwsIHNlbGYuZmlsZSA9IG9yaWdpbmFsLCBmaWxlCgogICAgZGVmIHdyaXRlKHNlbGYsIHRleHQpOgogICAgICAgIHNlbGYub3JpZ2luYWwud3JpdGUodGV4dCkKICAgICAgICBzZWxmLmZpbGUud3JpdGUodGV4dCkKICAgICAgICBzZWxmLmZpbGUuZmx1c2goKQogICAgICAgIHJldHVybiBsZW4odGV4dCkKCiAgICBkZWYgZmx1c2goc2VsZik6CiAgICAgICAgc2VsZi5vcmlnaW5hbC5mbHVzaCgpCiAgICAgICAgc2VsZi5maWxlLmZsdXNoKCkKCiAgICBkZWYgaXNhdHR5KHNlbGYpOgogICAgICAgIHJldHVybiBGYWxzZQoKCmRlZiBfcHJlcGFyZV9wYXRocyhjZmcpOgogICAgaWYgY2ZnWyJ1c2VfZHJpdmUiXToKICAgICAgICBpZiBub3QgUGF0aCgiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZSIpLmlzX2RpcigpOgogICAgICAgICAgICBmcm9tIGdvb2dsZS5jb2xhYiBpbXBvcnQgZHJpdmUKICAgICAgICAgICAgZHJpdmUubW91bnQoIi9jb250ZW50L2RyaXZlIikKICAgICAgICByb290ID0gUGF0aChjZmdbImRyaXZlX2RpciJdKQogICAgICAgIGlmIG5vdCBQYXRoKCIvY29udGVudC9kcml2ZS9NeURyaXZlIikuaXNfZGlyKCk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiR29vZ2xlIERyaXZlIGJlbHVtIHRlcnBhc2FuZy4gSmFsYW5rYW4ga2VtYmFsaSBtb3VudCBEcml2ZS4iKQogICAgZWxzZToKICAgICAgICByb290ID0gUGF0aChjZmcuZ2V0KCJ3b3JrX2RpciIsICIvY29udGVudC9MYXRpaElzeWFyYXRfVHJhaW5pbmdfSl9aIikpCiAgICB0aW1lc3RhbXAgPSBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5zdHJmdGltZSgiJVklbSVkXyVIJU0lU18lZl91dGMiKQogICAgcnVuID0gcm9vdCAvICJoYXNpbCIgLyBmInRyYWluaW5nX3t0aW1lc3RhbXB9X3tzdGFibGVfaGFzaChjZmcpWzo4XX0iCiAgICBwYXRocyA9IHsicm9vdCI6IHJvb3QsICJkYXRhIjogcm9vdCAvICJkYXRhc2V0cyIsICJjYWNoZSI6IHJvb3QgLyAiY2FjaGVfdjIiLAogICAgICAgICAgICAgIm5lZ2F0aXZlX3ZpZGVvcyI6IHJvb3QgLyAibmVnYXRpdmVfdmlkZW9zIiwgInJ1biI6IHJ1bn0KICAgIGZvciBwYXRoIGluIHBhdGhzLnZhbHVlcygpOgogICAgICAgIHBhdGgubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGF0aHNbImhhbmRfdGFzayJdID0gcGF0aHNbImRhdGEiXSAvICJtZWRpYXBpcGUiIC8gImhhbmRfbGFuZG1hcmtlcl92MS50YXNrIgogICAgcGF0aHNbImV4aXN0aW5nX3N0YXRpY19tb2RlbCJdID0gUGF0aChjZmcuZ2V0KCJleGlzdGluZ19zdGF0aWNfbW9kZWxfcGF0aCIpIG9yCiAgICAgICAgUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCAvICJiYXNlbGluZV9tb2RlbCIgLyAibW9kZWxfZmxvYXQxNi50ZmxpdGUiKQogICAgb3MuZW52aXJvblsiS0FHR0xFSFVCX0NBQ0hFIl0gPSBzdHIocGF0aHNbImRhdGEiXSAvICJrYWdnbGVodWIiKQogICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJURl9DUFBfTUlOX0xPR19MRVZFTCIsICIyIikKICAgIHJldHVybiBwYXRocwoKCmRlZiBfcHJlZmxpZ2h0KGNmZywgcGF0aHMpOgogICAgaW1wb3J0IHRlbnNvcmZsb3cgYXMgdGYKICAgIGltcG9ydCBtZWRpYXBpcGUgYXMgbXAKICAgIGdwdXMgPSB0Zi5jb25maWcubGlzdF9waHlzaWNhbF9kZXZpY2VzKCJHUFUiKQogICAgcHJpbnQoIlRlbnNvckZsb3c6IiwgdGYuX192ZXJzaW9uX18sICJNZWRpYVBpcGU6IiwgbXAuX192ZXJzaW9uX18sICJHUFU6IiwgZ3B1cykKICAgIGlmIGNmZ1sicmVxdWlyZV9ncHUiXSBhbmQgbm90IGdwdXM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJQaWxpaCBSdW50aW1lID4gQ2hhbmdlIHJ1bnRpbWUgdHlwZSA+IFQ0IEdQVSwga2VtdWRpYW4gUnVuIGFsbC4iKQogICAgZm9yIGdwdSBpbiBncHVzOgogICAgICAgIHRyeToKICAgICAgICAgICAgdGYuY29uZmlnLmV4cGVyaW1lbnRhbC5zZXRfbWVtb3J5X2dyb3d0aChncHUsIFRydWUpCiAgICAgICAgZXhjZXB0IFJ1bnRpbWVFcnJvcjoKICAgICAgICAgICAgcGFzcwogICAgaWYgbm90IGhhc2F0dHIobXAudGFza3MudmlzaW9uLCAiSGFuZExhbmRtYXJrZXIiKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk1lZGlhUGlwZSBUYXNrcyBIYW5kTGFuZG1hcmtlciB0aWRhayB0ZXJzZWRpYS4gSmFsYW5rYW4gc2VsIGluc3RhbGFzaS4iKQogICAgdGFyZ2V0ID0gcGF0aHNbImhhbmRfdGFzayJdCiAgICB0YXJnZXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGlmIG5vdCB0YXJnZXQuZXhpc3RzKCk6CiAgICAgICAgdGVtcG9yYXJ5ID0gdGFyZ2V0LndpdGhfc3VmZml4KCIucGFydCIpCiAgICAgICAgcHJpbnQoIk1lbmd1bmR1aCBtb2RlbCBNZWRpYVBpcGUgcmVzbWkgKHNla2FsaSwgZGlzaW1wYW4gZGkgRHJpdmUpLiIpCiAgICAgICAgdXJsbGliLnJlcXVlc3QudXJscmV0cmlldmUoSEFORF9UQVNLX1VSTCwgdGVtcG9yYXJ5KQogICAgICAgIHRlbXBvcmFyeS5yZXBsYWNlKHRhcmdldCkKICAgIGlmIHRhcmdldC5zdGF0KCkuc3Rfc2l6ZSA8IDFfMDAwXzAwMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkZpbGUgaGFuZF9sYW5kbWFya2VyLnRhc2sgdGlkYWsgbGVuZ2thcC4gSGFwdXMgZmlsZSB0ZXJzZWJ1dCBsYWx1IGphbGFua2FuIHVsYW5nLiIpCiAgICBqc29uX3dyaXRlKHBhdGhzWyJydW4iXSAvICJtZWRpYXBpcGVfYXNzZXQuanNvbiIsIHsKICAgICAgICAidXJsIjogSEFORF9UQVNLX1VSTCwgInNoYTI1NiI6IHNoYTI1Nl9maWxlKHRhcmdldCksICJzaXplX2J5dGVzIjogdGFyZ2V0LnN0YXQoKS5zdF9zaXplLAogICAgICAgICJtb2RlX3N0YXRpYyI6ICJJTUFHRSIsICJtb2RlX2R5bmFtaWMiOiAiVklERU8iLCAidHJhaW5hYmxlX2luX3RoaXNfd29ya2Zsb3ciOiBGYWxzZSwKICAgIH0pCgoKZGVmIF9wYWNrYWdlKHBhdGhzLCBzdW1tYXJ5KToKICAgIHJ1biA9IHBhdGhzWyJydW4iXQogICAganNvbl93cml0ZShydW4gLyAiUklOR0tBU0FOX0hBU0lMLmpzb24iLCBzdW1tYXJ5KQogICAgaW52ZW50b3J5ID0gW10KICAgIGZvciBwYXRoIGluIHNvcnRlZChydW4ucmdsb2IoIioiKSk6CiAgICAgICAgaWYgcGF0aC5pc19maWxlKCkgYW5kIHBhdGgubmFtZSAhPSAiU0hBMjU2U1VNUy5jc3YiOgogICAgICAgICAgICBpbnZlbnRvcnkuYXBwZW5kKHsicGF0aCI6IHBhdGgucmVsYXRpdmVfdG8ocnVuKS5hc19wb3NpeCgpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAic2l6ZV9ieXRlcyI6IHBhdGguc3RhdCgpLnN0X3NpemUsICJzaGEyNTYiOiBzaGEyNTZfZmlsZShwYXRoKX0pCiAgICB3cml0ZV9jc3YocnVuIC8gIlNIQTI1NlNVTVMuY3N2IiwgaW52ZW50b3J5KQogICAgYXJjaGl2ZSA9IHJ1bi5wYXJlbnQgLyBmIntydW4ubmFtZX1fTEVOR0tBUC56aXAiCiAgICB0ZW1wb3JhcnkgPSBhcmNoaXZlLndpdGhfc3VmZml4KCIuemlwLnBhcnRpYWwiKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUodGVtcG9yYXJ5LCAidyIsIGNvbXByZXNzaW9uPXppcGZpbGUuWklQX0RFRkxBVEVELCBjb21wcmVzc2xldmVsPTYpIGFzIG91dHB1dDoKICAgICAgICBmb3IgcGF0aCBpbiBzb3J0ZWQocnVuLnJnbG9iKCIqIikpOgogICAgICAgICAgICBpZiBwYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgICAgIG91dHB1dC53cml0ZShwYXRoLCBQYXRoKHJ1bi5uYW1lKSAvIHBhdGgucmVsYXRpdmVfdG8ocnVuKSkKICAgIHRlbXBvcmFyeS5yZXBsYWNlKGFyY2hpdmUpCiAgICBwcmludCgiXG5GT0xERVIgSEFTSUwgUEVSTUFORU46IiwgcnVuKQogICAgcHJpbnQoIlNBVFUgWklQIExFTkdLQVA6IiwgYXJjaGl2ZSkKICAgIHByaW50KCJVa3VyYW4gWklQOiIsIHJvdW5kKGFyY2hpdmUuc3RhdCgpLnN0X3NpemUgLyAxMDI0KioyLCAyKSwgIk1pQiIpCiAgICByZXR1cm4gYXJjaGl2ZQoKCmRlZiBydW5fYWxsKGNvbmZpZz1Ob25lKToKICAgIGNmZyA9IHsqKkRFRkFVTFRfQ09ORklHLCAqKihjb25maWcgb3Ige30pfQogICAgaWYgY2ZnWyJzZXF1ZW5jZV9sZW5ndGgiXSA8IDggb3IgY2ZnWyJkeW5hbWljX3NhbXBsZV9mcHMiXSA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNlcXVlbmNlX2xlbmd0aCA+PSA4IGRhbiBkeW5hbWljX3NhbXBsZV9mcHMgPiAwIGRpcGVybHVrYW4uIikKICAgIHBhdGhzID0gX3ByZXBhcmVfcGF0aHMoY2ZnKQogICAgc3VtbWFyeSA9IHsic3RhdHVzIjogIlJVTk5JTkciLCAicnVuX2RpcmVjdG9yeSI6IHN0cihwYXRoc1sicnVuIl0pLAogICAgICAgICAgICAgICAic3RhcnRlZF91dGMiOiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKSwgImNvbmZpZyI6IGNmZywKICAgICAgICAgICAgICAgImNsYWltcyI6IHsKICAgICAgICAgICAgICAgICAgICJwZXJmZWN0X2FjY3VyYWN5X2d1YXJhbnRlZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAgICAgICJtb2JpbGVfaW50ZWdyYXRpb25fcGVyZm9ybWVkIjogRmFsc2UsCiAgICAgICAgICAgICAgICAgICAic3RhdGljX2FuZF9keW5hbWljX2FyZV9zZXBhcmF0ZV9tb2RlbHMiOiBUcnVlLAogICAgICAgICAgICAgICAgICAgInRlc3RfcmVzdWx0c19hcmVfbm90X3VzZWRfZm9yX21vZGVsX3NlbGVjdGlvbiI6IFRydWUsCiAgICAgICAgICAgICAgICAgICAic3RhdGljMjRfd2VpZ2h0c19wcmVzZXJ2ZWQiOiBUcnVlLAogICAgICAgICAgICAgICAgICAgIkhGX3N0YXRpY19zY29yZXNfYXJlX2RpYWdub3N0aWNfbm90X2luZGVwZW5kZW50X3Rlc3QiOiBUcnVlLAogICAgICAgICAgICAgICB9fQogICAgZXJyb3IgPSBOb25lCiAgICBsb2dfcGF0aCA9IHBhdGhzWyJydW4iXSAvICJjb25zb2xlLmxvZyIKICAgIHdpdGggbG9nX3BhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGxvZzoKICAgICAgICB3aXRoIGNvbnRleHRsaWIucmVkaXJlY3Rfc3Rkb3V0KF9UZWUoc3lzLnN0ZG91dCwgbG9nKSksIGNvbnRleHRsaWIucmVkaXJlY3Rfc3RkZXJyKF9UZWUoc3lzLnN0ZGVyciwgbG9nKSk6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHByaW50KCJPdXRwdXQgcGVybWFuZW46IiwgcGF0aHNbInJ1biJdKQogICAgICAgICAgICAgICAgcHJpbnQoIlRhcmdldCBha3VyYXNpIGFkYWxhaCBzYXNhcmFuIGVrc3BlcmltZW4sIGJ1a2FuIGphbWluYW4uIikKICAgICAgICAgICAgICAgIGpzb25fd3JpdGUocGF0aHNbInJ1biJdIC8gImNvbmZpZy5qc29uIiwgY2ZnKQogICAgICAgICAgICAgICAgc291cmNlX2RpciA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKICAgICAgICAgICAgICAgIHNuYXBzaG90ID0gcGF0aHNbInJ1biJdIC8gInNvdXJjZV9jb2RlIgogICAgICAgICAgICAgICAgc25hcHNob3QubWtkaXIoKQogICAgICAgICAgICAgICAgZm9yIG5hbWUgaW4gKCJ0cmFpbl9hbGwucHkiLCAidHJhaW5pbmdfY29tbW9uLnB5IiwgInN0YXRpY190cmFpbmluZy5weSIsICJkeW5hbWljX3RyYWluaW5nLnB5IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAicmVxdWlyZW1lbnRzX2NvbGFiLnR4dCIsICJSRUFETUUubWQiLCAiTEFQT1JBTl9NT0RFTF9EQU5fUkVOQ0FOQV9JTlRFR1JBU0kubWQiKToKICAgICAgICAgICAgICAgICAgICBzb3VyY2UgPSBzb3VyY2VfZGlyIC8gbmFtZQogICAgICAgICAgICAgICAgICAgIGlmIHNvdXJjZS5leGlzdHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgc2h1dGlsLmNvcHkyKHNvdXJjZSwgc25hcHNob3QgLyBuYW1lKQogICAgICAgICAgICAgICAgX3ByZWZsaWdodChjZmcsIHBhdGhzKQogICAgICAgICAgICAgICAgcmVjb3JkX2Vudmlyb25tZW50KHBhdGhzWyJydW4iXSkKICAgICAgICAgICAgICAgIGZyb20gc3RhdGljX3RyYWluaW5nIGltcG9ydCBydW5fc3RhdGljCiAgICAgICAgICAgICAgICBmcm9tIGR5bmFtaWNfdHJhaW5pbmcgaW1wb3J0IHJ1bl9keW5hbWljCiAgICAgICAgICAgICAgICBwcmludCgiXG5bMS8yXSBQRVJUQUhBTktBTiBNT0RFTCAyNCBIVVJVRiBEQU4gRVZBTFVBU0kgQ1JPUCBNRURJQVBJUEUiKQogICAgICAgICAgICAgICAgc3VtbWFyeVsic3RhdGljMjQiXSA9IHJ1bl9zdGF0aWMoY2ZnLCBwYXRocykKICAgICAgICAgICAgICAgIGpzb25fd3JpdGUocGF0aHNbInJ1biJdIC8gIlJJTkdLQVNBTl9IQVNJTC5qc29uIiwgc3VtbWFyeSkKICAgICAgICAgICAgICAgICMgRnJlZSB0aGUgc3RhdGljIHRyYWluaW5nIGdyYXBoIGJlZm9yZSBjcmVhdGluZyB0aGUgdGVtcG9yYWwgbW9kZWwuCiAgICAgICAgICAgICAgICBpbXBvcnQgZ2MKICAgICAgICAgICAgICAgIGltcG9ydCB0ZW5zb3JmbG93IGFzIHRmCiAgICAgICAgICAgICAgICB0Zi5rZXJhcy5iYWNrZW5kLmNsZWFyX3Nlc3Npb24oKQogICAgICAgICAgICAgICAgZ2MuY29sbGVjdCgpCiAgICAgICAgICAgICAgICBwcmludCgiXG5bMi8yXSBUUkFJTklORyBHRVJBS0FOIEovWiArIE9USEVSIikKICAgICAgICAgICAgICAgIHN1bW1hcnlbImR5bmFtaWNfanoiXSA9IHJ1bl9keW5hbWljKGNmZywgcGF0aHMsIHN1bW1hcnlbInN0YXRpYzI0Il0pCiAgICAgICAgICAgICAgICBzaXplcyA9IFt7InBhdGgiOiBwLnJlbGF0aXZlX3RvKHBhdGhzWyJydW4iXSkuYXNfcG9zaXgoKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAic2l6ZV9ieXRlcyI6IHAuc3RhdCgpLnN0X3NpemUsICJzaXplX21pYiI6IHAuc3RhdCgpLnN0X3NpemUgLyAxMDI0KioyfQogICAgICAgICAgICAgICAgICAgICAgICAgZm9yIHAgaW4gc29ydGVkKHBhdGhzWyJydW4iXS5yZ2xvYigiKi50ZmxpdGUiKSldCiAgICAgICAgICAgICAgICBzdW1tYXJ5WyJleHBvcnRlZF9tb2RlbF9zaXplcyJdID0gc2l6ZXMKICAgICAgICAgICAgICAgIHN1bW1hcnlbInNpemVfYnVkZ2V0X21pYl9wZXJfbW9kZWwiXSA9IGNmZ1sibWF4X21vZGVsX21pYiJdCiAgICAgICAgICAgICAgICBzdW1tYXJ5WyJhbGxfZXhwb3J0c193aXRoaW5fc2l6ZV9idWRnZXQiXSA9IGFsbCgKICAgICAgICAgICAgICAgICAgICBpdGVtWyJzaXplX21pYiJdIDw9IGNmZ1sibWF4X21vZGVsX21pYiJdIGZvciBpdGVtIGluIHNpemVzKQogICAgICAgICAgICAgICAgc3VtbWFyeVsic3RhdHVzIl0gPSAiQ09NUExFVEVEIgogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgICAgIGVycm9yID0gZXhjCiAgICAgICAgICAgICAgICBzdW1tYXJ5WyJzdGF0dXMiXSA9ICJGQUlMRUQiCiAgICAgICAgICAgICAgICBzdW1tYXJ5WyJlcnJvciJdID0geyJ0eXBlIjogdHlwZShleGMpLl9fbmFtZV9fLCAibWVzc2FnZSI6IHN0cihleGMpfQogICAgICAgICAgICAgICAgdHJhY2ViYWNrLnByaW50X2V4YygpCiAgICAgICAgICAgICAgICAocGF0aHNbInJ1biJdIC8gImZhaWx1cmVfdHJhY2ViYWNrLnR4dCIpLndyaXRlX3RleHQodHJhY2ViYWNrLmZvcm1hdF9leGMoKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICAgICAgZmluYWxseToKICAgICAgICAgICAgICAgIHN1bW1hcnlbImZpbmlzaGVkX3V0YyJdID0gZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCkKICAgICAgICAgICAgICAgIHByaW50KCJcblNUQVRVUzoiLCBzdW1tYXJ5WyJzdGF0dXMiXSkKICAgIGFyY2hpdmUgPSBfcGFja2FnZShwYXRocywgc3VtbWFyeSkKICAgIGlmIGNmZy5nZXQoInJlc3VsdF9wb2ludGVyIik6CiAgICAgICAganNvbl93cml0ZShjZmdbInJlc3VsdF9wb2ludGVyIl0sIHsic3RhdHVzIjogc3VtbWFyeVsic3RhdHVzIl0sICJ6aXBfcGF0aCI6IHN0cihhcmNoaXZlKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInJ1bl9kaXJlY3RvcnkiOiBzdHIocGF0aHNbInJ1biJdKX0pCiAgICBpZiBjZmdbImF1dG9fZG93bmxvYWQiXToKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZyb20gZ29vZ2xlLmNvbGFiIGltcG9ydCBmaWxlcwogICAgICAgICAgICBmaWxlcy5kb3dubG9hZChzdHIoYXJjaGl2ZSkpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBkb3dubG9hZF9lcnJvcjoKICAgICAgICAgICAgcHJpbnQoIlVuZHVoIFpJUCBzZWNhcmEgbWFudWFsIGRhcmkgRHJpdmU6IiwgYXJjaGl2ZSwgInwiLCBkb3dubG9hZF9lcnJvcikKICAgIGlmIGVycm9yIGlzIG5vdCBOb25lOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlRyYWluaW5nIGJlbHVtIHNlbGVzYWkuIEJ1a3RpIGRhbiBoYXNpbCBzZW1lbnRhcmEgYWRhIGRpIHthcmNoaXZlfSIpIGZyb20gZXJyb3IKICAgIHByaW50KCJTZWxlc2FpLiBQZXJpa3NhIFJJTkdLQVNBTl9IQVNJTC5qc29uIGRhbiBtb2RlbF9jYXJkIHNlYmVsdW0gbWVtaWxpaCBtb2RlbC4iKQogICAgcmV0dXJuIHsqKnN1bW1hcnksICJ6aXBfcGF0aCI6IHN0cihhcmNoaXZlKX0KCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgdHlwZT1QYXRoLCBoZWxwPSJKU09OIGtvbmZpZ3VyYXNpIG9wc2lvbmFsIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBydW5fYWxsKGpzb24ubG9hZHMoYXJncy5jb25maWcucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKSBpZiBhcmdzLmNvbmZpZyBlbHNlIE5vbmUpCg==","sha256":"f186a87dc3be5a2afbbf03f17f47ecac46a02b251a4e80333a86c91aae01d025"},"training_common.py":{"base64":"IiIiU2hhcmVkLCBhdWRpdGFibGUgdXRpbGl0aWVzIGZvciB0aGUgQ29sYWIgdHJhaW5pbmcgd29ya2Zsb3cuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHBsYXRmb3JtCmltcG9ydCByYW5kb20KaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCgoKZGVmIF9zZXJpYWxpemFibGUodmFsdWUpOgogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgZGljdCk6CiAgICAgICAgcmV0dXJuIHtzdHIoayk6IF9zZXJpYWxpemFibGUodikgZm9yIGssIHYgaW4gdmFsdWUuaXRlbXMoKX0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIChsaXN0LCB0dXBsZSkpOgogICAgICAgIHJldHVybiBbX3NlcmlhbGl6YWJsZSh2KSBmb3IgdiBpbiB2YWx1ZV0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIG5wLm5kYXJyYXkpOgogICAgICAgIHJldHVybiBfc2VyaWFsaXphYmxlKHZhbHVlLnRvbGlzdCgpKQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgbnAuZ2VuZXJpYyk6CiAgICAgICAgcmV0dXJuIF9zZXJpYWxpemFibGUodmFsdWUuaXRlbSgpKQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgUGF0aCk6CiAgICAgICAgcmV0dXJuIHN0cih2YWx1ZSkKICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIGZsb2F0KSBhbmQgbm90IG5wLmlzZmluaXRlKHZhbHVlKToKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIHZhbHVlCgoKZGVmIGpzb25fd3JpdGUocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRleHQgPSBqc29uLmR1bXBzKF9zZXJpYWxpemFibGUodmFsdWUpLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yLCBhbGxvd19uYW49RmFsc2UpCiAgICB3aXRoIHRlbXBmaWxlLk5hbWVkVGVtcG9yYXJ5RmlsZShtb2RlPSJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgZGlyPXBhdGgucGFyZW50LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3VmZml4PSIucGFydGlhbCIsIGRlbGV0ZT1GYWxzZSkgYXMgc3RyZWFtOgogICAgICAgIHN0cmVhbS53cml0ZSh0ZXh0ICsgIlxuIikKICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKHN0cmVhbS5uYW1lKQogICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKICAgIHJldHVybiBwYXRoCgoKZGVmIHN0YWJsZV9oYXNoKHZhbHVlKToKICAgIHBheWxvYWQgPSBqc29uLmR1bXBzKF9zZXJpYWxpemFibGUodmFsdWUpLCBzb3J0X2tleXM9VHJ1ZSwgc2VwYXJhdG9ycz0oIiwiLCAiOiIpLAogICAgICAgICAgICAgICAgICAgICAgICAgZW5zdXJlX2FzY2lpPUZhbHNlLCBhbGxvd19uYW49RmFsc2UpLmVuY29kZSgidXRmLTgiKQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHBheWxvYWQpLmhleGRpZ2VzdCgpCgoKZGVmIHNoYTI1Nl9maWxlKHBhdGgpOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBQYXRoKHBhdGgpLm9wZW4oInJiIikgYXMgc3RyZWFtOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTogc3RyZWFtLnJlYWQoMTAyNCAqIDEwMjQpLCBiIiIpOgogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGJsb2NrKQogICAgcmV0dXJuIGRpZ2VzdC5oZXhkaWdlc3QoKQoKCmRlZiB3cml0ZV9jc3YocGF0aCwgcm93cyk6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcm93cyA9IGxpc3Qocm93cykKICAgIGNvbHVtbnMgPSBsaXN0KGRpY3QuZnJvbWtleXMoa2V5IGZvciByb3cgaW4gcm93cyBmb3Iga2V5IGluIHJvdykpCiAgICB3aXRoIHBhdGgub3BlbigidyIsIG5ld2xpbmU9IiIsIGVuY29kaW5nPSJ1dGYtOC1zaWciKSBhcyBzdHJlYW06CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoc3RyZWFtLCBmaWVsZG5hbWVzPWNvbHVtbnMpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdyh7azoganNvbi5kdW1wcyhfc2VyaWFsaXphYmxlKHYpLCBlbnN1cmVfYXNjaWk9RmFsc2UpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZSh2LCAoZGljdCwgbGlzdCwgdHVwbGUsIG5wLm5kYXJyYXkpKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVsc2UgX3NlcmlhbGl6YWJsZSh2KSBmb3IgaywgdiBpbiByb3cuaXRlbXMoKX0pCiAgICByZXR1cm4gcGF0aAoKCmRlZiBzZWVkX2V2ZXJ5dGhpbmcoc2VlZCk6CiAgICByYW5kb20uc2VlZChzZWVkKQogICAgbnAucmFuZG9tLnNlZWQoc2VlZCkKICAgIGltcG9ydCB0ZW5zb3JmbG93IGFzIHRmCiAgICB0Zi5rZXJhcy51dGlscy5zZXRfcmFuZG9tX3NlZWQoc2VlZCkKICAgIHRyeToKICAgICAgICB0Zi5jb25maWcuZXhwZXJpbWVudGFsLmVuYWJsZV9vcF9kZXRlcm1pbmlzbSgpCiAgICBleGNlcHQgKEF0dHJpYnV0ZUVycm9yLCBSdW50aW1lRXJyb3IpOgogICAgICAgIHBhc3MKCgpkZWYgc3BsaXRfcmVjb3JkcyhyZWNvcmRzLCB2YWxpZGF0aW9uX2ZyYWN0aW9uLCB0ZXN0X2ZyYWN0aW9uLCBzZWVkKToKICAgICIiIlNwbGl0IG9yaWdpbmFsIGdyb3VwcywgbmV2ZXIgZGVyaXZlZCBpbWFnZXMvd2luZG93cy4gUmVxdWlyZXMgPj0zIGdyb3Vwcy9jbGFzcy4KCiAgICBBIGdyb3VwIGNhbiBjb250YWluIHNldmVyYWwgY2xhc3NlcyAoZS5nLiBhIGNvbXBsZXRlIG5lZ2F0aXZlIHJlY29yZGluZyBzZXNzaW9uKS4KICAgIEl0cyBtb3N0IGZyZXF1ZW50IGxhYmVsIGRldGVybWluZXMgdGhlIHN0cmF0aWZpY2F0aW9uIGJ1Y2tldDsgZ3JvdXAgaW50ZWdyaXR5IHdpbnMKICAgIG92ZXIgZXhhY3QgZnJhY3Rpb25zLiBGcmFjdGlvbnMgYW5kIHJlYWxpemVkIGNvdW50cyBhcmUgcmV0YWluZWQgaW4gbWFuaWZlc3RzLgogICAgIiIiCiAgICBpZiBub3QgMCA8PSB2YWxpZGF0aW9uX2ZyYWN0aW9uIDwgMSBvciBub3QgMCA8PSB0ZXN0X2ZyYWN0aW9uIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTcGxpdCBmcmFjdGlvbnMgbXVzdCBiZSBpbiBbMCwxKS4iKQogICAgaWYgdmFsaWRhdGlvbl9mcmFjdGlvbiArIHRlc3RfZnJhY3Rpb24gPj0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUcmFpbmluZyBmcmFjdGlvbiBtdXN0IGJlIHBvc2l0aXZlLiIpCiAgICBncm91cHMgPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIHJlY29yZCBpbiByZWNvcmRzOgogICAgICAgIGlmIG5vdCByZWNvcmQuZ2V0KCJncm91cF9pZCIpIG9yICJsYWJlbCIgbm90IGluIHJlY29yZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRWFjaCBzb3VyY2UgcmVjb3JkIG5lZWRzIGdyb3VwX2lkIGFuZCBsYWJlbC4iKQogICAgICAgIGdyb3Vwc1tzdHIocmVjb3JkWyJncm91cF9pZCJdKV0uYXBwZW5kKGRpY3QocmVjb3JkKSkKICAgIGJ1Y2tldHMgPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIGdyb3VwLCBtZW1iZXJzIGluIGdyb3Vwcy5pdGVtcygpOgogICAgICAgIGxhYmVscyA9IFtzdHIoaXRlbVsibGFiZWwiXSkgZm9yIGl0ZW0gaW4gbWVtYmVyc10KICAgICAgICBwcmltYXJ5ID0gc29ydGVkKHNldChsYWJlbHMpLCBrZXk9bGFtYmRhIHg6ICgtbGFiZWxzLmNvdW50KHgpLCB4KSlbMF0KICAgICAgICBidWNrZXRzW3ByaW1hcnldLmFwcGVuZChncm91cCkKICAgIGFzc2lnbm1lbnQgPSB7fQogICAgZm9yIGxhYmVsLCBpZHMgaW4gc29ydGVkKGJ1Y2tldHMuaXRlbXMoKSk6CiAgICAgICAgaWRzID0gc29ydGVkKGlkcykKICAgICAgICByZXF1aXJlZCA9IDEgKyBpbnQodmFsaWRhdGlvbl9mcmFjdGlvbiA+IDApICsgaW50KHRlc3RfZnJhY3Rpb24gPiAwKQogICAgICAgIGlmIGxlbihpZHMpIDwgcmVxdWlyZWQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDbGFzcyB7bGFiZWx9OiBuZWVkID49IHtyZXF1aXJlZH0gZGlzdGluY3Qgc291cmNlIGdyb3VwcywgZ290IHtsZW4oaWRzKX0uIikKICAgICAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoaW50KHN0YWJsZV9oYXNoKFtpbnQoc2VlZCksIGxhYmVsXSlbOjE2XSwgMTYpKQogICAgICAgIGlkcyA9IFtpZHNbaV0gZm9yIGkgaW4gcm5nLnBlcm11dGF0aW9uKGxlbihpZHMpKV0KICAgICAgICBuX3Rlc3QgPSBtYXgoMSwgcm91bmQobGVuKGlkcykgKiB0ZXN0X2ZyYWN0aW9uKSkgaWYgdGVzdF9mcmFjdGlvbiBlbHNlIDAKICAgICAgICBuX3ZhbCA9IG1heCgxLCByb3VuZChsZW4oaWRzKSAqIHZhbGlkYXRpb25fZnJhY3Rpb24pKSBpZiB2YWxpZGF0aW9uX2ZyYWN0aW9uIGVsc2UgMAogICAgICAgIHdoaWxlIG5fdGVzdCArIG5fdmFsID49IGxlbihpZHMpOgogICAgICAgICAgICBpZiBuX3Rlc3QgPiAxIGFuZCBuX3Rlc3QgPj0gbl92YWw6CiAgICAgICAgICAgICAgICBuX3Rlc3QgLT0gMQogICAgICAgICAgICBlbGlmIG5fdmFsID4gMToKICAgICAgICAgICAgICAgIG5fdmFsIC09IDEKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDYW5ub3QgcmVzZXJ2ZSB0cmFpbmluZyBncm91cHMgZm9yIHtsYWJlbH0uIikKICAgICAgICBmb3IgaSwgZ3JvdXAgaW4gZW51bWVyYXRlKGlkcyk6CiAgICAgICAgICAgIGFzc2lnbm1lbnRbZ3JvdXBdID0gKCJ0ZXN0IiBpZiBpIDwgbl90ZXN0IGVsc2UgInZhbGlkYXRpb24iCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIGkgPCBuX3Rlc3QgKyBuX3ZhbCBlbHNlICJ0cmFpbiIpCiAgICBvdXRwdXQgPSBbeyoqcmVjb3JkLCAic3BsaXQiOiBhc3NpZ25tZW50W3N0cihyZWNvcmRbImdyb3VwX2lkIl0pXX0gZm9yIHJlY29yZCBpbiByZWNvcmRzXQogICAgc2VlbiA9IGRlZmF1bHRkaWN0KHNldCkKICAgIGZvciByZWNvcmQgaW4gb3V0cHV0OgogICAgICAgIHNlZW5bcmVjb3JkWyJzcGxpdCJdXS5hZGQoc3RyKHJlY29yZFsiZ3JvdXBfaWQiXSkpCiAgICBhc3NlcnQgbm90IChzZWVuWyJ0cmFpbiJdICYgc2VlblsidmFsaWRhdGlvbiJdIG9yIHNlZW5bInRyYWluIl0gJiBzZWVuWyJ0ZXN0Il0KICAgICAgICAgICAgICAgIG9yIHNlZW5bInZhbGlkYXRpb24iXSAmIHNlZW5bInRlc3QiXSkKICAgIHJldHVybiBvdXRwdXQKCgpkZWYgY2FjaGVkX25weihjYWNoZV9kaXIsIGtleSwgYnVpbGRlcik6CiAgICAiIiJDb250ZW50LWFkZHJlc3NlZCBOUFogY2FjaGUsIGluY2x1ZGluZyBhIHZhbGlkYXRlZCBlbWJlZGRlZCBzaWduYXR1cmUuIiIiCiAgICBjYWNoZV9kaXIgPSBQYXRoKGNhY2hlX2RpcikKICAgIGNhY2hlX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBzaWduYXR1cmUgPSBzdGFibGVfaGFzaChrZXkpCiAgICBwYXRoID0gY2FjaGVfZGlyIC8gZiJ7c2lnbmF0dXJlfS5ucHoiCiAgICBpZiBwYXRoLmV4aXN0cygpOgogICAgICAgIHRyeToKICAgICAgICAgICAgd2l0aCBucC5sb2FkKHBhdGgsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgYXJjaGl2ZToKICAgICAgICAgICAgICAgIGlmIHN0cihhcmNoaXZlWyJfY2FjaGVfc2lnbmF0dXJlIl0uaXRlbSgpKSAhPSBzaWduYXR1cmU6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ2FjaGUgc2lnbmF0dXJlIG1pc21hdGNoLiIpCiAgICAgICAgICAgICAgICByZXR1cm4ge25hbWU6IGFyY2hpdmVbbmFtZV0gZm9yIG5hbWUgaW4gYXJjaGl2ZS5maWxlcyBpZiBuYW1lICE9ICJfY2FjaGVfc2lnbmF0dXJlIn0KICAgICAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IsIEtleUVycm9yLCBFT0ZFcnJvcik6CiAgICAgICAgICAgIHByaW50KCJDYWNoZSBpbnZhbGlkLCByZWJ1aWxkaW5nOiIsIHBhdGgubmFtZSkKICAgIGFycmF5cyA9IHtuYW1lOiBucC5hc2FycmF5KHZhbHVlKSBmb3IgbmFtZSwgdmFsdWUgaW4gYnVpbGRlcigpLml0ZW1zKCl9CiAgICBpZiBhbnkodmFsdWUuZHR5cGUuaGFzb2JqZWN0IGZvciB2YWx1ZSBpbiBhcnJheXMudmFsdWVzKCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk9iamVjdCBhcnJheXMgYXJlIGZvcmJpZGRlbiBpbiByZXByb2R1Y2libGUgY2FjaGVzLiIpCiAgICB3aXRoIHRlbXBmaWxlLk5hbWVkVGVtcG9yYXJ5RmlsZShkaXI9Y2FjaGVfZGlyLCBzdWZmaXg9Ii5ucHoiLCBkZWxldGU9RmFsc2UpIGFzIHN0cmVhbToKICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKHN0cmVhbS5uYW1lKQogICAgdHJ5OgogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQodGVtcG9yYXJ5LCBfY2FjaGVfc2lnbmF0dXJlPW5wLmFycmF5KHNpZ25hdHVyZSksICoqYXJyYXlzKQogICAgICAgIHRlbXBvcmFyeS5yZXBsYWNlKHBhdGgpCiAgICBmaW5hbGx5OgogICAgICAgIHRlbXBvcmFyeS51bmxpbmsobWlzc2luZ19vaz1UcnVlKQogICAganNvbl93cml0ZShjYWNoZV9kaXIgLyBmIntzaWduYXR1cmV9Lmpzb24iLCB7InNpZ25hdHVyZSI6IHNpZ25hdHVyZSwgImtleSI6IGtleSwKICAgICAgICAgICAgICAgImFycmF5cyI6IHtrOiB7InNoYXBlIjogbGlzdCh2LnNoYXBlKSwgImR0eXBlIjogc3RyKHYuZHR5cGUpfSBmb3IgaywgdiBpbiBhcnJheXMuaXRlbXMoKX19KQogICAgcmV0dXJuIGFycmF5cwoKCmRlZiBjbGFzc2lmaWNhdGlvbl9yZXBvcnRfYXJ0aWZhY3RzKHlfdHJ1ZSwgc2NvcmVzLCBsYWJlbHMsIG91dF9kaXIsIHByZWZpeCk6CiAgICBmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgKGFjY3VyYWN5X3Njb3JlLCBjbGFzc2lmaWNhdGlvbl9yZXBvcnQsIGNvbmZ1c2lvbl9tYXRyaXgsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByZWNpc2lvbl9yZWNhbGxfZnNjb3JlX3N1cHBvcnQpCiAgICBpbXBvcnQgbWF0cGxvdGxpYgogICAgbWF0cGxvdGxpYi51c2UoIkFnZyIpCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CiAgICBvdXRfZGlyID0gUGF0aChvdXRfZGlyKQogICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB5X3RydWUgPSBucC5hc2FycmF5KHlfdHJ1ZSwgZHR5cGU9bnAuaW50NjQpCiAgICBzY29yZXMgPSBucC5hc2FycmF5KHNjb3JlcykKICAgIGlmIGxlbih5X3RydWUpID09IDA6CiAgICAgICAgcmVwb3J0ID0geyJzYW1wbGVzIjogMCwgImFjY3VyYWN5IjogTm9uZSwgInJlYXNvbiI6ICJObyBldmFsdWFibGUgc2FtcGxlcy4ifQogICAgICAgIGpzb25fd3JpdGUob3V0X2RpciAvIGYie3ByZWZpeH1fbWV0cmljcy5qc29uIiwgcmVwb3J0KQogICAgICAgIHJldHVybiByZXBvcnQKICAgIGlmIHNjb3Jlcy5zaGFwZSAhPSAobGVuKHlfdHJ1ZSksIGxlbihsYWJlbHMpKSBvciBub3QgbnAuaXNmaW5pdGUoc2NvcmVzKS5hbGwoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJQcmVkaWN0aW9uIHNjb3JlcyBoYXZlIGludmFsaWQgc2hhcGUgb3Igbm9uZmluaXRlIHZhbHVlcy4iKQogICAgcHJlZGljdGVkID0gc2NvcmVzLmFyZ21heChheGlzPTEpCiAgICBpbmRpY2VzID0gbGlzdChyYW5nZShsZW4obGFiZWxzKSkpCiAgICBkZXRhaWwgPSBjbGFzc2lmaWNhdGlvbl9yZXBvcnQoeV90cnVlLCBwcmVkaWN0ZWQsIGxhYmVscz1pbmRpY2VzLCB0YXJnZXRfbmFtZXM9bGFiZWxzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHplcm9fZGl2aXNpb249MCwgb3V0cHV0X2RpY3Q9VHJ1ZSkKICAgIHByZWNpc2lvbiwgcmVjYWxsLCBmMSwgXyA9IHByZWNpc2lvbl9yZWNhbGxfZnNjb3JlX3N1cHBvcnQoCiAgICAgICAgeV90cnVlLCBwcmVkaWN0ZWQsIGxhYmVscz1pbmRpY2VzLCBhdmVyYWdlPSJtYWNybyIsIHplcm9fZGl2aXNpb249MCkKICAgIHByZXNlbnQgPSBzb3J0ZWQoc2V0KHlfdHJ1ZS50b2xpc3QoKSkpCiAgICBwcmVzZW50X3ByZWNpc2lvbiwgcHJlc2VudF9yZWNhbGwsIHByZXNlbnRfZjEsIF8gPSBwcmVjaXNpb25fcmVjYWxsX2ZzY29yZV9zdXBwb3J0KAogICAgICAgIHlfdHJ1ZSwgcHJlZGljdGVkLCBsYWJlbHM9cHJlc2VudCwgYXZlcmFnZT0ibWFjcm8iLCB6ZXJvX2RpdmlzaW9uPTApCiAgICBtYXRyaXggPSBjb25mdXNpb25fbWF0cml4KHlfdHJ1ZSwgcHJlZGljdGVkLCBsYWJlbHM9aW5kaWNlcykKICAgIHJlcG9ydCA9IHsic2FtcGxlcyI6IGxlbih5X3RydWUpLCAiYWNjdXJhY3kiOiBmbG9hdChhY2N1cmFjeV9zY29yZSh5X3RydWUsIHByZWRpY3RlZCkpLAogICAgICAgICAgICAgICJtYWNyb19wcmVjaXNpb24iOiBmbG9hdChwcmVjaXNpb24pLCAibWFjcm9fcmVjYWxsIjogZmxvYXQocmVjYWxsKSwKICAgICAgICAgICAgICAibWFjcm9fZjEiOiBmbG9hdChmMSksICJtYWNyb19hdmVyYWdlX2RlZmluaXRpb24iOiAiYWxsIGRlY2xhcmVkIG91dHB1dCBjbGFzc2VzIiwKICAgICAgICAgICAgICAiZXZhbHVhdGVkX3RydWVfbGFiZWxzIjogW2xhYmVsc1tpXSBmb3IgaSBpbiBwcmVzZW50XSwKICAgICAgICAgICAgICAibWFjcm9fcHJlY2lzaW9uX3ByZXNlbnRfY2xhc3NlcyI6IGZsb2F0KHByZXNlbnRfcHJlY2lzaW9uKSwKICAgICAgICAgICAgICAibWFjcm9fcmVjYWxsX3ByZXNlbnRfY2xhc3NlcyI6IGZsb2F0KHByZXNlbnRfcmVjYWxsKSwKICAgICAgICAgICAgICAibWFjcm9fZjFfcHJlc2VudF9jbGFzc2VzIjogZmxvYXQocHJlc2VudF9mMSksICJwZXJfY2xhc3MiOiBkZXRhaWx9CiAgICBqc29uX3dyaXRlKG91dF9kaXIgLyBmIntwcmVmaXh9X21ldHJpY3MuanNvbiIsIHJlcG9ydCkKICAgIHdyaXRlX2NzdihvdXRfZGlyIC8gZiJ7cHJlZml4fV9wZXJfY2xhc3MuY3N2IiwgWwogICAgICAgIHsibGFiZWwiOiBsYWJlbCwgKipkZXRhaWxbbGFiZWxdfSBmb3IgbGFiZWwgaW4gbGFiZWxzXSkKICAgIHdyaXRlX2NzdihvdXRfZGlyIC8gZiJ7cHJlZml4fV9wcmVkaWN0aW9ucy5jc3YiLCBbCiAgICAgICAgeyJyb3ciOiBpLCAidHJ1ZV9sYWJlbCI6IGxhYmVsc1tpbnQoeV90cnVlW2ldKV0sICJwcmVkaWN0ZWRfbGFiZWwiOiBsYWJlbHNbaW50KHByZWRpY3RlZFtpXSldLAogICAgICAgICAiY29uZmlkZW5jZSI6IGZsb2F0KHNjb3Jlc1tpXS5tYXgoKSksICJjb3JyZWN0IjogYm9vbCh5X3RydWVbaV0gPT0gcHJlZGljdGVkW2ldKX0KICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oeV90cnVlKSldKQogICAgd3JpdGVfY3N2KG91dF9kaXIgLyBmIntwcmVmaXh9X2NvbmZ1c2lvbl9tYXRyaXguY3N2IiwgWwogICAgICAgIHsidHJ1ZV9sYWJlbCI6IGxhYmVsLCAqKmRpY3QoemlwKGxhYmVscywgbWF0cml4W2ldLnRvbGlzdCgpKSl9IGZvciBpLCBsYWJlbCBpbiBlbnVtZXJhdGUobGFiZWxzKV0pCiAgICBzaXplID0gbWF4KDUsIG1pbigxMywgbGVuKGxhYmVscykgKiAuNDMpKQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPShzaXplLCBzaXplKSkKICAgIGltID0gYXguaW1zaG93KG1hdHJpeCwgY21hcD0iQmx1ZXMiKQogICAgYXguc2V0KHh0aWNrcz1pbmRpY2VzLCB5dGlja3M9aW5kaWNlcywgeHRpY2tsYWJlbHM9bGFiZWxzLCB5dGlja2xhYmVscz1sYWJlbHMsCiAgICAgICAgICAgeGxhYmVsPSJQcmVkaWtzaSIsIHlsYWJlbD0iTGFiZWwgYXNsaSIsIHRpdGxlPWYie3ByZWZpeH0gfCBhY2N1cmFjeSB7cmVwb3J0WydhY2N1cmFjeSddOi4yJX0iKQogICAgZmlnLmNvbG9yYmFyKGltLCBheD1heCwgZnJhY3Rpb249LjA0NikKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgZmlnLnNhdmVmaWcob3V0X2RpciAvIGYie3ByZWZpeH1fY29uZnVzaW9uX21hdHJpeC5wbmciLCBkcGk9MTYwKQogICAgcGx0LmNsb3NlKGZpZykKICAgIHJldHVybiByZXBvcnQKCgpkZWYgZXhwb3J0X3RmbGl0ZShtb2RlbCwgb3V0X2RpciwgbmFtZSwgcmVwcmVzZW50YXRpdmVfaW5wdXRzPU5vbmUsCiAgICAgICAgICAgICAgICAgIHZhbGlkYXRpb25faW5wdXRzPU5vbmUsIHZhbGlkYXRpb25fbGFiZWxzPU5vbmUpOgogICAgIiIiRXhwb3J0IGJ1aWx0aW4tb25seSBtb2RlbHMgYW5kIGNoZWNrIHBhcml0eSBvbiBWQUxJREFUSU9OLCBub3QgZmluYWwgdGVzdC4iIiIKICAgIGltcG9ydCB0ZW5zb3JmbG93IGFzIHRmCiAgICBvdXRfZGlyID0gUGF0aChvdXRfZGlyKQogICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBzYW1wbGVzID0gdmFsaWRhdGlvbl9pbnB1dHMgaWYgdmFsaWRhdGlvbl9pbnB1dHMgaXMgbm90IE5vbmUgZWxzZSByZXByZXNlbnRhdGl2ZV9pbnB1dHMKICAgIGlmIHNhbXBsZXMgaXMgTm9uZSBvciBsZW4oc2FtcGxlcykgPT0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBvcnQgcGFyaXR5IG5lZWRzIG5vbmVtcHR5IHZhbGlkYXRpb24gaW5wdXRzLiIpCiAgICBzYW1wbGVzID0gbnAuYXNhcnJheShzYW1wbGVzWzoxMjhdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAga2VyYXNfc2NvcmVzID0gbW9kZWwucHJlZGljdChzYW1wbGVzLCBiYXRjaF9zaXplPW1pbig2NCwgbGVuKHNhbXBsZXMpKSwgdmVyYm9zZT0wKQogICAgc3VtbWFyaWVzID0ge30KICAgIGZvciB2YXJpYW50IGluICgiZmxvYXQzMiIsICJmbG9hdDE2Iik6CiAgICAgICAgY29udmVydGVyID0gdGYubGl0ZS5URkxpdGVDb252ZXJ0ZXIuZnJvbV9rZXJhc19tb2RlbChtb2RlbCkKICAgICAgICBjb252ZXJ0ZXIudGFyZ2V0X3NwZWMuc3VwcG9ydGVkX29wcyA9IFt0Zi5saXRlLk9wc1NldC5URkxJVEVfQlVJTFRJTlNdCiAgICAgICAgaWYgdmFyaWFudCA9PSAiZmxvYXQxNiI6CiAgICAgICAgICAgIGNvbnZlcnRlci5vcHRpbWl6YXRpb25zID0gW3RmLmxpdGUuT3B0aW1pemUuREVGQVVMVF0KICAgICAgICAgICAgY29udmVydGVyLnRhcmdldF9zcGVjLnN1cHBvcnRlZF90eXBlcyA9IFt0Zi5mbG9hdDE2XQogICAgICAgIHBheWxvYWQgPSBjb252ZXJ0ZXIuY29udmVydCgpCiAgICAgICAgcGF0aCA9IG91dF9kaXIgLyBmIntuYW1lfV97dmFyaWFudH0udGZsaXRlIgogICAgICAgIHBhdGgud3JpdGVfYnl0ZXMocGF5bG9hZCkKICAgICAgICBpbnRlcnByZXRlciA9IHRmLmxpdGUuSW50ZXJwcmV0ZXIobW9kZWxfY29udGVudD1wYXlsb2FkLCBudW1fdGhyZWFkcz0yKQogICAgICAgIGludGVycHJldGVyLmFsbG9jYXRlX3RlbnNvcnMoKQogICAgICAgIGlucHV0X2luZm8gPSBpbnRlcnByZXRlci5nZXRfaW5wdXRfZGV0YWlscygpWzBdCiAgICAgICAgb3V0cHV0X2luZm8gPSBpbnRlcnByZXRlci5nZXRfb3V0cHV0X2RldGFpbHMoKVswXQogICAgICAgIGV4cGVjdGVkID0gWzEsICpzYW1wbGVzLnNoYXBlWzE6XV0KICAgICAgICBpZiBpbnB1dF9pbmZvWyJkdHlwZSJdICE9IG5wLmZsb2F0MzIgb3Igb3V0cHV0X2luZm9bImR0eXBlIl0gIT0gbnAuZmxvYXQzMjoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFeHBlY3RlZCBmbG9hdDMyIGlucHV0L291dHB1dCB0ZW5zb3JzLiIpCiAgICAgICAgaWYgbGlzdChpbnB1dF9pbmZvWyJzaGFwZSJdKSAhPSBleHBlY3RlZDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVW5leHBlY3RlZCBURkxpdGUgaW5wdXQ6IHtpbnB1dF9pbmZvWydzaGFwZSddfSAhPSB7ZXhwZWN0ZWR9LiIpCiAgICAgICAgbGl0ZV9zY29yZXMgPSBbXQogICAgICAgIGZvciByb3cgaW4gc2FtcGxlczoKICAgICAgICAgICAgaW50ZXJwcmV0ZXIuc2V0X3RlbnNvcihpbnB1dF9pbmZvWyJpbmRleCJdLCByb3dbTm9uZV0pCiAgICAgICAgICAgIGludGVycHJldGVyLmludm9rZSgpCiAgICAgICAgICAgIGxpdGVfc2NvcmVzLmFwcGVuZChpbnRlcnByZXRlci5nZXRfdGVuc29yKG91dHB1dF9pbmZvWyJpbmRleCJdKVswXS5jb3B5KCkpCiAgICAgICAgbGl0ZV9zY29yZXMgPSBucC5hc2FycmF5KGxpdGVfc2NvcmVzKQogICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShsaXRlX3Njb3JlcykuYWxsKCk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTm9uZmluaXRlIFRGTGl0ZSBwcmVkaWN0aW9ucy4iKQogICAgICAgIGFncmVlbWVudCA9IGZsb2F0KG5wLm1lYW4oa2VyYXNfc2NvcmVzLmFyZ21heCgxKSA9PSBsaXRlX3Njb3Jlcy5hcmdtYXgoMSkpKQogICAgICAgIGRpZmZlcmVuY2UgPSBmbG9hdChucC5tYXgobnAuYWJzKGtlcmFzX3Njb3JlcyAtIGxpdGVfc2NvcmVzKSkpCiAgICAgICAgb3BlcmF0b3JzID0gc29ydGVkKHtvcFsib3BfbmFtZSJdIGZvciBvcCBpbiBpbnRlcnByZXRlci5fZ2V0X29wc19kZXRhaWxzKCl9KQogICAgICAgIGlmIGFueShvcC5zdGFydHN3aXRoKCJGbGV4IikgZm9yIG9wIGluIG9wZXJhdG9ycyk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRXhwb3J0IHJlcXVpcmVzIFNlbGVjdCBURiBPcHMsIG5vdCByZWFkeSBmb3IgdGhlIGN1cnJlbnQgRmx1dHRlciBpbnRlcnByZXRlci4iKQogICAgICAgIHJlcG9ydCA9IHsicGF0aCI6IHN0cihwYXRoKSwgInNpemVfYnl0ZXMiOiBsZW4ocGF5bG9hZCksICJzaXplX21pYiI6IGxlbihwYXlsb2FkKSAvIDEwMjQqKjIsCiAgICAgICAgICAgICAgICAgICJzaGEyNTYiOiBoYXNobGliLnNoYTI1NihwYXlsb2FkKS5oZXhkaWdlc3QoKSwgImlucHV0X3NoYXBlIjogaW5wdXRfaW5mb1sic2hhcGUiXS50b2xpc3QoKSwKICAgICAgICAgICAgICAgICAgIm91dHB1dF9zaGFwZSI6IG91dHB1dF9pbmZvWyJzaGFwZSJdLnRvbGlzdCgpLCAiaW5wdXRfZHR5cGUiOiAiZmxvYXQzMiIsCiAgICAgICAgICAgICAgICAgICJvdXRwdXRfZHR5cGUiOiAiZmxvYXQzMiIsICJvcGVyYXRvcnMiOiBvcGVyYXRvcnMsICJwYXJpdHlfc2FtcGxlcyI6IGxlbihzYW1wbGVzKSwKICAgICAgICAgICAgICAgICAgImxhYmVsX2FncmVlbWVudCI6IGFncmVlbWVudCwgIm1heGltdW1fYWJzb2x1dGVfc2NvcmVfZGlmZmVyZW5jZSI6IGRpZmZlcmVuY2UsCiAgICAgICAgICAgICAgICAgICJwYXJpdHlfcGFzc2VkIjogYWdyZWVtZW50ID49IC45OSBhbmQgZGlmZmVyZW5jZSA8PSAuMDUsCiAgICAgICAgICAgICAgICAgICJwYXJpdHlfc3BsaXQiOiAidmFsaWRhdGlvbiJ9CiAgICAgICAgaWYgdmFsaWRhdGlvbl9sYWJlbHMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGxhYmVscyA9IG5wLmFzYXJyYXkodmFsaWRhdGlvbl9sYWJlbHNbOmxlbihzYW1wbGVzKV0pCiAgICAgICAgICAgIHJlcG9ydFsidmFsaWRhdGlvbl9zYW1wbGVfYWNjdXJhY3kiXSA9IGZsb2F0KG5wLm1lYW4obGl0ZV9zY29yZXMuYXJnbWF4KDEpID09IGxhYmVscykpCiAgICAgICAganNvbl93cml0ZShvdXRfZGlyIC8gZiJ7bmFtZX1fe3ZhcmlhbnR9X2V4cG9ydC5qc29uIiwgcmVwb3J0KQogICAgICAgIHN1bW1hcmllc1t2YXJpYW50XSA9IHJlcG9ydAogICAgICAgIGlmIG5vdCByZXBvcnRbInBhcml0eV9wYXNzZWQiXToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie25hbWV9L3t2YXJpYW50fTogVEZMaXRlIHBhcml0eSBmYWlsZWQuIFNlZSBleHBvcnQgcmVwb3J0LiIpCiAgICByZXR1cm4gc3VtbWFyaWVzCgoKZGVmIHJlY29yZF9lbnZpcm9ubWVudChvdXRfZGlyKToKICAgIG91dF9kaXIgPSBQYXRoKG91dF9kaXIpCiAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG5hbWVzID0gWyJ0ZW5zb3JmbG93IiwgImtlcmFzIiwgIm51bXB5IiwgIm1lZGlhcGlwZSIsICJvcGVuY3YtY29udHJpYi1weXRob24iLAogICAgICAgICAgICAgInBhbmRhcyIsICJweWFycm93IiwgInNjaWtpdC1sZWFybiIsICJtYXRwbG90bGliIiwgIlBpbGxvdyIsICJrYWdnbGVodWIiXQogICAgcGFja2FnZXMgPSB7fQogICAgZm9yIG5hbWUgaW4gbmFtZXM6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwYWNrYWdlc1tuYW1lXSA9IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKG5hbWUpCiAgICAgICAgZXhjZXB0IGltcG9ydGxpYi5tZXRhZGF0YS5QYWNrYWdlTm90Rm91bmRFcnJvcjoKICAgICAgICAgICAgcGFja2FnZXNbbmFtZV0gPSBOb25lCiAgICByZXBvcnQgPSB7InB5dGhvbiI6IHN5cy52ZXJzaW9uLCAicGxhdGZvcm0iOiBwbGF0Zm9ybS5wbGF0Zm9ybSgpLCAicGFja2FnZXMiOiBwYWNrYWdlcywKICAgICAgICAgICAgICAicmVjb3JkZWRfYXRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCl9CiAgICBqc29uX3dyaXRlKG91dF9kaXIgLyAiZW52aXJvbm1lbnQuanNvbiIsIHJlcG9ydCkKICAgIHJlc3VsdCA9IHN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwgIi1tIiwgInBpcCIsICJmcmVlemUiXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlKQogICAgKG91dF9kaXIgLyAicmVxdWlyZW1lbnRzX2FjdHVhbC50eHQiKS53cml0ZV90ZXh0KHJlc3VsdC5zdGRvdXQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gcmVwb3J0Cg==","sha256":"cee2da020d622682ec4510e9e77aac2ff3f2f357866f37f9bc5c391ea505886a"},"static_training.py":{"base64":"...","sha256":"02bad446651e38308e4e76c6c3a607ab55c84a0515381b2a28f14144674f5b24"},"dynamic_training.py":{"base64":"...","sha256":"fa99131606bba40c748edbad465eb13e8668aebf2d747b9c72fab9eaf42d878e"},"requirements_colab.txt":{"base64":"IyBUZW5zb3JGbG93IEdQVSBkaXNlZGlha2FuIHJ1bnRpbWUgQ29sYWIuIFZlcnNpIGFrdHVhbCBkaWNhdGF0IGRpIGhhc2lsLgojIE1lZGlhUGlwZSBpbmkgbWVtcHVueWFpIHdoZWVsIExpbnV4IGdlbmVyaWsgUHl0aG9uIDMgdW50dWsgQ29sYWIgbW9kZXJuLgptZWRpYXBpcGU9PTAuMTAuMzIKb3BlbmN2LWNvbnRyaWItcHl0aG9uPT00LjExLjAuODYKa2FnZ2xlaHViPT0wLjMuMTMKbnVtcHk+PTEuMjYsPDIuMwpwYW5kYXM+PTIuMiw8MwpweWFycm93Pj0xOSw8MjQKc2Npa2l0LWxlYXJuPj0xLjYsPDIKbWF0cGxvdGxpYj49My45LDw0ClBpbGxvdz49MTEsPDEzCnJlcXVlc3RzPj0yLjMyLDwzCg==","sha256":"713f87832b296063792bcf8533006f62a3f6717becca1cdedeb022fc4ae06f0b"},"README.md":{"base64":"IyBUcmFpbmluZyBKIGRhbiBaIGRpIEdvb2dsZSBDb2xhYgoKIyMgWWFuZyBkaWxha3VrYW4KCk1vZGVsIDI0IGh1cnVmIHlhbmcgc2VrYXJhbmcgZGlwYWthaSBhcGxpa2FzaSAqKmRpcGVydGFoYW5rYW4qKiwgYnVrYW4gZGlidWF0IHVsYW5nLgpOb3RlYm9vayBtZW1iYXdhIHNhbGluYW4gbW9kZWwgdGVyc2VidXQsIHVrdXJhbiAqKjMyNi4xMDggYnl0ZSoqLCBTSEEtMjU2CmA3NGY1MmFiMGQ2ZjA5MWIzNzYwMGNlNjVjNGI3Nzk3YTEwMmM2ZTI2MTEyYjZjMmU0ODI0NDgzYjE2ODE0MGJjYC4KClNhdHUgbm90ZWJvb2sgbWVuZ2VyamFrYW4gZHVhIHRhaGFwOgoKMS4gKipFdmFsdWFzaSBtb2RlbCAyNCBodXJ1ZiB5YW5nIGFkYS4qKiBNZW5ndWppIE1OSVNUIGRhbiBmb3RvIEh1Z2dpbmcgRmFjZSBkZW5nYW4KICAgY3JvcCBNZWRpYVBpcGUgSU1BR0UsIHBhZGRpbmcgY29udGV4dC90aWdodCwgaW5wdXQgMOKAkzI1NSwgZGFuIHBlbmdnYWJ1bmdhbgogICBwcmVkaWtzaSBzZXBlcnRpIGFwbGlrYXNpLiBUaWRhayBhZGEgdHJhaW5pbmcsIHBlbmdnYW50aWFuIGJvYm90LCBhdGF1IGdyYWZpawogICB0cmFpbmluZyBiYXJ1IHVudHVrIG1vZGVsIGluaS4KMi4gKipUcmFpbmluZyBtb2RlbCBnZXJha2FuIGJhcnUgSi9aL09USEVSLioqIE1lbWFrYWkgdXJ1dGFuIGxhbmRtYXJrIE1lZGlhUGlwZQogICBWSURFTyBkYW4gR1JVIGtlY2lsLiBPcmlnaW5hbCB2aWRlbyBkaXBpc2Foa2FuIHNlYmVsdW0gYXVnbWVudGFzaS4gQ2hlY2twb2ludAogICBkaXBpbGloIGJlcmRhc2Fya2FuIHZhbGlkYXRpb24gbG9zcywgbGFsdSB0ZXN0IGRpbGFrdWthbiBzZXRlbGFoIHBlbWlsaWhhbi4KCkFwbGlrYXNpIEZsdXR0ZXIgdGlkYWsgZGl1YmFoIGF0YXUgZGktYnVpbGQgb2xlaCBub3RlYm9vayBpbmkuIEltcGxlbWVudGFzaSBKL1oKZGkgYXBsaWthc2kgZGlsYWt1a2FuIHNldGVsYWggaGFzaWwgdHJhaW5pbmcgZGlwZXJpa3NhLgoKIyMgQ2FyYSBtZW5qYWxhbmthbgoKMS4gQnVrYSAqKlRSQUlOSU5HX0pfWl9DT0xBQi5pcHluYioqIG1lbGFsdWkgR29vZ2xlIENvbGFiID4gRmlsZSA+IFVwbG9hZCBub3RlYm9vay4KICAgTm90ZWJvb2sgc3VkYWggbWVtdWF0IGtvZGUgcGVuZGFtcGluZyBkYW4gbW9kZWwgMjQgaHVydWYuIFRpZGFrIHBlcmx1IG1lbmd1bmdnYWgKICAgZmlsZSBgLnB5YCBzYXR1IHBlciBzYXR1IGF0YXUgYmVyZ2FudHVuZyBwYWRhIGtvZGUgR2l0SHViIHlhbmcgYmVydWJhaC4KMi4gUGlsaWggKipSdW50aW1lID4gQ2hhbmdlIHJ1bnRpbWUgdHlwZSA+IFQ0IEdQVSoqLgozLiBQaWxpaCAqKlJ1bnRpbWUgPiBSdW4gYWxsKiogZGFuIGl6aW5rYW4gcGVtYXNhbmdhbiBHb29nbGUgRHJpdmUuCjQuIFR1bmdndSBoaW5nZ2Egc3RhdHVzIGBDT01QTEVURURgLiBHcmFmaWssIGxhcG9yYW4sIGRhbiBtb2RlbCBkaWtlbWFzIG1lbmphZGkKICAgKipzYXR1IFpJUCB5YW5nIG90b21hdGlzIGRpdGF3YXJrYW4gdW50dWsgZGl1bmR1aCoqLiBFa3N0cmFrIFpJUCB1bnR1ayBtZW1wZXJvbGVoCiAgIHNhdHUgZm9sZGVyIGxlbmdrYXAuIEppa2EgYnJvd3NlciBtZW5haGFuIHVuZHVoYW4sIGFtYmlsIFpJUCB5YW5nIHNhbWEgZGkgRHJpdmUuCgpUcmFpbmluZyBiZXJqYWxhbiBkYWxhbSBwcm9zZXMgUHl0aG9uIGJhcnUgc2V0ZWxhaCBpbnN0YWxhc2kgZGVwZW5kZW5jeSBhZ2FyCnBha2V0IG5hdGl2ZSBsYW1hIGRpIGtlcm5lbCBub3RlYm9vayB0aWRhayB0ZXJiYXdhLiBMb2cgdGV0YXAgdGFtcGlsIGRpIENvbGFiLgpWZXJzaSBUZW5zb3JGbG93IEdQVSBiYXdhYW4gcnVudGltZSBkaXBlcnRhaGFua2FuOyB2ZXJzaSBha3R1YWwgc2VsdXJ1aCBwYWtldApkaXNpbXBhbiBkYWxhbSBoYXNpbC4gVGlkYWsgcGVybHUgbWVuamFsYW5rYW4gbm90ZWJvb2sgc2FhdCBpbmkgZGkgbGFwdG9wLgoKIyMgTG9rYXNpIHBlbnlpbXBhbmFuIERyaXZlCgpgYGB0ZXh0Ck15RHJpdmUvTGF0aWhJc3lhcmF0X1RyYWluaW5nX0pfWi8KICBkYXRhc2V0cy8gICAgICAgICAgICAgZGF0YXNldCBkYW4gbW9kZWwgTWVkaWFQaXBlLCBkaWd1bmFrYW4gdWxhbmcKICBjYWNoZV92Mi8gICAgICAgICAgICAgY3JvcCBzZXJ0YSBsYW5kbWFyayBkZW5nYW4gaWRlbnRpdGFzIGtvZGUvY29uZmlnL2RhdGEKICBuZWdhdGl2ZV92aWRlb3MvICAgICAgb3BzaW9uYWwsIHJla2FtYW4gZ2VyYWthbiBsYWluIG1pbGlrIHRpbQogIGhhc2lsLwogICAgdHJhaW5pbmdfPHdha3R1Pl88Y29uZmlnPi8KICAgIHRyYWluaW5nXzx3YWt0dT5fPGNvbmZpZz5fTEVOR0tBUC56aXAKYGBgCgpWaWRlbyBTaWdOTiBzZWtpdGFyICoqNCwxNyBHQiB1bnR1ayB1bmR1aGFuKiouIFNpYXBrYW4gcnVhbmcgdGFtYmFoYW4gdW50dWsKZWtzdHJha3NpLCBmb3RvLCBkYW4gY2FjaGUuIERhdGFzZXQgbWVudGFoIGJlcnVrdXJhbiBiZXNhciB0ZXRhcCBkaSBEcml2ZSBkYW4KdGlkYWsgaWt1dCBaSVAgYnVrdGkuIFJ1biBiZXJpa3V0bnlhIG1lbWFrYWkgZmlsZSB5YW5nIHN1ZGFoIHRlcnNlZGlhLiBDYWNoZQpkaXBha2FpIGtlbWJhbGkgaGFueWEgamlrYSBzdW1iZXIsIGtvbmZpZ3VyYXNpIHBlbXJvc2VzYW4sIG1vZGVsIE1lZGlhUGlwZSwKdmVyc2kgZGVwZW5kZW5jeSwgZGFuIGtvZGUgeWFuZyByZWxldmFuIGNvY29rLgoKIyMgRGF0YXNldAoKfCBTdW1iZXIgfCBQZW5nZ3VuYWFuIHwKfCAtLS0gfCAtLS0gfAp8IFtTaWduIExhbmd1YWdlIE1OSVNUXShodHRwczovL3d3dy5rYWdnbGUuY29tL2RhdGFzZXRzL2RhdGFtdW5nZS9zaWduLWxhbmd1YWdlLW1uaXN0KSB8IEV2YWx1YXNpIG1vZGVsIDI0IGh1cnVmIHBhZGEgb2ZmaWNpYWwgdGVzdC4gVGlkYWsgZGlsYXRpaCB1bGFuZy4gfAp8IFtGb3RvIEFTTCBIdWdnaW5nIEZhY2VdKGh0dHBzOi8vaHVnZ2luZ2ZhY2UuY28vZGF0YXNldHMvTWFyeHVsaWEvYXNsX3NpZ25fbGFuZ3VhZ2VzX2FscGhhYmV0c192MDMpIHwgRGlhZ25vc3RpayBtb2RlbCAyNCBodXJ1ZiBtZW5nZ3VuYWthbiBjcm9wIE1lZGlhUGlwZSB5YW5nIHNlc3VhaSBhcGxpa2FzaS4gSi9aIGRpa2VsdWFya2FuLiB8CnwgW1NpZ05OIEovWl0oaHR0cHM6Ly93d3cua2FnZ2xlLmNvbS9kYXRhc2V0cy9zaWdubnRlYW0vYXNsLXNpZ24tbGFuZ3VhZ2UtYWxwaGFiZXQtdmlkZW9zLWoteikgfCBUcmFpbmluZywgdmFsaWRhdGlvbiwgZGFuIHRlc3QgcGVuZ2VuYWwgZ2VyYWthbiBKL1ouIE1hbmlmZXN0IHB1YmxpayBtZW11YXQgNzEyIEFWSSBzZWJlbHVtIHBlbWVyaWtzYWFuIGR1cGxpa2FzaS4gSnVtbGFoIGFrdHVhbCBkaWNhdGF0IHNhYXQgcnVuLiB8CgoqKkZvdG8gSHVnZ2luZyBGYWNlIGJ1a2FuIHRlc3QgaW5kZXBlbmRlbiBiYXJ1LioqIE1vZGVsIDI0IGh1cnVmIHNlYmVsdW1ueWEgc3VkYWgKZGlsYXRpaCBtZW5nZ3VuYWthbiBzdW1iZXIgdGVyc2VidXQsIHNlbWVudGFyYSBrZWFuZ2dvdGFhbiBjYWNoZSB0cmFpbmluZyBsYW1hCmJlbHVtIGRhcGF0IGRpdmVyaWZpa2FzaS4gTGFwb3JhbiBiYXJ1IG1lbmFtYWlueWEgZGlhZ25vc3RpaywgYnVrYW4gYnVrdGkKZ2VuZXJhbGlzYXNpLiBBbmdrYW55YSB0aWRhayBib2xlaCBsYW5nc3VuZyBkaWJhbmRpbmdrYW4gZGVuZ2FuIDQ0LDI2NCUgbGFtYQpzZWJhZ2FpIGtlbmFpa2FuIHlhbmcgc2V0YXJhIGthcmVuYSBzYW1wZWwsIGNyb3AsIGRhbiBkZW5vbWluYXRvciBiZXJiZWRhLgoKIyMgS2VsYXMgT1RIRVIgZGFuIGJhdGFzIGhhc2lsIGVrc3BlcmltZW4KCkphbmdhbiBtZWxhdGloIGhhbnlhIEogdmVyc3VzIFoga2FyZW5hIHNldGlhcCBnZXJha2FuIGxhaW4gYWthbiBkaXBha3NhIG1hc3VrCmtlIHNhbGFoIHNhdHVueWEuIFJ1biBkZWZhdWx0IG1lbmFtYmFoa2FuICoqT1RIRVIgc2ludGV0aXMgYmVydXBhIHBvc2UgZGlhbSoqCnlhbmcgZGlidWF0IGRhcmkgdmlkZW8gc3VtYmVyIGRhbiB0ZXRhcCBiZXJhZGEgcGFkYSBzcGxpdCB2aWRlbyBpbmR1a255YS4KSGFzaWwgc2ludGV0aXMgZGlsYXBvcmthbiB0ZXJwaXNhaCBkYXJpIHZpZGVvIGFzbGkuIEluaSBidWthbiBwZW5nZ2FudGkgZGF0YQpnZXJha2FuIGxhaW4geWFuZyBiZW5hci1iZW5hciBkaXJla2FtLgoKVW50dWsgZWtzcGVyaW1lbiB5YW5nIGxlYmloIGt1YXQsIHNlYmVsdW0gUnVuIGFsbCBtYXN1a2thbiByZWthbWFuIGdlcmFrYW4gYmlhc2EsCnBlcnBpbmRhaGFuIHBvc2UsIGRhbiBwb3NlIGRpYW0ga2U6CgpgYGB0ZXh0Cm5lZ2F0aXZlX3ZpZGVvcy8KICBvcmFuZ18wMS8gICB2aWRlbyBnZXJha2FuIHNlbGFpbiBKL1oKICBvcmFuZ18wMi8gICB2aWRlbyBnZXJha2FuIHNlbGFpbiBKL1oKICBvcmFuZ18wMy8gICB2aWRlbyBnZXJha2FuIHNlbGFpbiBKL1oKYGBgCgpHdW5ha2FuIHNldGlkYWtueWEgKip0aWdhIGZvbGRlciBvcmFuZy9zZXNpKiogYWdhciB0cmFpbmluZywgdmFsaWRhdGlvbiwgZGFuIHRlc3QKZGFwYXQgZGlwaXNhaGthbi4gU2VtdWEgcmVrYW1hbiBvcmFuZyB5YW5nIHNhbWEgc2ViYWlrbnlhIG1hc3VrIHNhdHUgZm9sZGVyLgpKYW5nYW4gbWVuYXJ1aCB2aWRlbyBKL1ogZGkgZm9sZGVyIG5lZ2F0aWYuIFJla2FtYW4gdGFucGEgdGFuZ2FuIHRldGFwIGRpY2F0YXQKc2ViYWdhaSBkZXRla3NpIGdhZ2FsLCBidWthbiBkaXBhbHN1a2FuIG1lbmphZGkgc2VxdWVuY2UgdGFuZ2FuIHZhbGlkLgoKKipUYXJnZXQgOTksNSUgbWVydXBha2FuIHNhc2FyYW4sIGJ1a2FuIGphbWluYW4uKiogVGlkYWsgYWRhIHBlbmNhcmlhbiBrb25maWd1cmFzaQpiZXJkYXNhcmthbiB0ZXN0IHNhbXBhaSBhbmdrYSB0ZXJsaWhhdCBzZW1wdXJuYS4gTmlsYWkgc2VlZCB5YW5nIGxlYmloIGJlc2FyCnRpZGFrIG1lbWJ1YXQgbW9kZWwgbGViaWggcGludGFyLiBgc2VlZGAgbWVuZ2F0dXIgdHJhaW5pbmcsIHNlZGFuZ2thbgpgc3BsaXRfc2VlZGAgbWVuamFnYSBwZW1pc2FoYW4gc3VtYmVyIHRldGFwIGtvbnNpc3Rlbi4KCkRhdGFzZXQgU2lnTk4gdGlkYWsgbWVueWVkaWFrYW4gaWRlbnRpdGFzIHBlcmFnYSB5YW5nIHRlcnZlcmlmaWthc2kuIFBlbWJhZ2lhbgp2aWRlbyB0aWRhayBtZW1idWt0aWthbiBwZW5ndWppYW4gcGFkYSBvcmFuZyB5YW5nIGJlbmFyLWJlbmFyIGJhcnUuIEFrdXJhc2kgdmlkZW8KcGVuZGVrIGp1Z2EgdGlkYWsgc2FtYSBkZW5nYW4ga2V0ZXBhdGFuIGRldGVrc2kgYXdhbC9ha2hpciBnZXJha2FuIHBhZGEga2FtZXJhCmtvbnRpbnUuIFN0YXR1cyBtb2RlbCB0ZXRhcCBtZW1lcmx1a2FuIHVqaSBwZW5nZ3VuYSBkYW4ga2FtZXJhIG55YXRhLgoKIyMgSXNpIGJ1a3RpIGhhc2lsCgotIGBSSU5HS0FTQU5fSEFTSUwuanNvbmAsIGtvbmZpZ3VyYXNpLCBsb2cgY29uc29sZSwgd2FrdHUgcnVuLCB2ZXJzaSBkZXBlbmRlbmN5LAogIHNuYXBzaG90IGtvZGUsIHNlcnRhIGRhZnRhciBTSEEtMjU2IHNlbXVhIGFydGVmYWsuCi0gTW9kZWwgMjQgaHVydWYgYXNsaSBiZXNlcnRhIG1ldGFkYXRhIGRhbiBwZW1lcmlrc2FhbiBpZGVudGl0YXNueWEuCi0gSGFzaWwgZGlhZ25vc3RpayBNTklTVC9IdWdnaW5nIEZhY2UsIHByZWRpa3NpIHBlciBnYW1iYXIsIGNvbmZ1c2lvbiBtYXRyaXgsCiAgcHJlY2lzaW9uL3JlY2FsbC9GMSBwZXIgaHVydWYsIHRpbmdrYXQgZGV0ZWtzaSBkYW4gcGVub2xha2FuLCBzZXJ0YSBjb250b2ggY3JvcC4KLSBNb2RlbCBKL1ovT1RIRVIgYC5rZXJhc2AsIFRGTGl0ZSBmbG9hdDMyIGRhbiBmbG9hdDE2LCB1a3VyYW4gc2ViZW5hcm55YSwKICBsYWJlbCwga29udHJhayBmaXR1ciwgZGFuIGJ1a3RpIGtlc2VzdWFpYW4gcHJlZGlrc2kgZWtzcG9yIHBhZGEgdmFsaWRhdGlvbi4KLSBLdXJ2YSB0cmFpbi92YWxpZGF0aW9uLCBDU1Ygc2V0aWFwIGVwb2NoLCBlcG9jaCB0ZXJiYWlrLCBsYXBvcmFuIGZpbmFsIHRlc3QsCiAgY29uZnVzaW9uIG1hdHJpeCwgcHJlZGlrc2kgcGVyIHZpZGVvLCBrZXNhbGFoYW4sIGRhbiBjb3ZlcmFnZSBkZXRla3NpLgotIE1hbmlmZXN0IHN1bWJlci9zcGxpdC9kdXBsaWthc2ksIGxhbmRtYXJrIG1lbnRhaCBiZXJ0aW1lc3RhbXAsIGZpdHVyIHNlcXVlbmNlLAogIGNvbnRvaCBsaW50YXNhbiwgZGFuIG1vZGVsIGNhcmQgeWFuZyBtZW5qZWxhc2thbiBrZXRlcmJhdGFzYW4uCgpMYXBvcmFuIHRyYWluaW5nL3Rlc3QgR1JVIG1lbWFrYWkgY2hlY2twb2ludCBLZXJhcyB0ZXJiYWlrLiBURkxpdGUgZGlwZXJpa3NhCm1lbGFsdWkgcGFyaXR5IHBhZGEgdmFsaWRhdGlvbjsgamFuZ2FuIG1lbnllYnV0bnlhIHBlbmd1a3VyYW4gZnVsbC10ZXN0IFRGTGl0ZQpkaSBIUC4gS2VnYWdhbGFuIGRldGVrc2kgdGV0YXAgdGVybGloYXQgZGFsYW0gZGVub21pbmF0b3IgbWV0cmlrIGVuZC10by1lbmQuCgpKaWthIHNhbGFoIHNhdHUgdGFoYXAgZ2FnYWwsIFpJUCBoYXNpbCBzZW1lbnRhcmEgZGFuIHRyYWNlYmFjayB0ZXRhcCBkaWJ1YXQuCmBGQUlMRURgIGJ1a2FuIGhhc2lsIHRyYWluaW5nIHNlbGVzYWkuIEd1bmFrYW4gZm9sZGVyIGJ1a3RpIGl0dSB1bnR1ayBkaWFnbm9zaXMuCgojIyBGaWxlIHN1bWJlcgoKYHRyYWluX2FsbC5weWAgYWRhbGFoIGVudHJ5IHBvaW50LCBgc3RhdGljX3RyYWluaW5nLnB5YCBtZW5nZXZhbHVhc2kgbW9kZWwgeWFuZwphZGEsIGBkeW5hbWljX3RyYWluaW5nLnB5YCBtZWxhdGloIEovWiwgZGFuIGB0cmFpbmluZ19jb21tb24ucHlgIG1lbnllZGlha2FuCnNwbGl0L2NhY2hlL2xhcG9yYW4vZWtzcG9yLiBOb3RlYm9vayBtZW5nZW1hcyBzZW11YW55YSBiZXNlcnRhIGBiYXNlbGluZV9tb2RlbC9gLgpgZ2VuZXJhdGVfbm90ZWJvb2sucHlgIG1lbXBlcmJhcnVpIG5vdGVib29rIHNldGVsYWgga29kZSBzdW1iZXIgZGl1YmFoLgoKUGVyYmFpa2FuIGJvYm90IDI0IGh1cnVmIGtlbGFrIG1lbWJ1dHVoa2FuIGNoZWNrcG9pbnQgdHJhaW5pbmcgc2VwZXJ0aQpgZmluYWwua2VyYXNgLiBGaWxlIFRGTGl0ZSBkaXBlcnRhaGFua2FuIHVudHVrIGluZmVyZW5zaSBkYW4gYnVrYW4gY2hlY2twb2ludApmaW5lLXR1bmluZy4gTm90ZWJvb2sgaW5pIHRpZGFrIGRpYW0tZGlhbSBtZW5nZ2FudGlueWEgZGVuZ2FuIENOTiBiYXJ1Lgo=","sha256":"188a776f8d3007b18b1e8082bc7bd9676b235de7e87153f1354fb7aca9a84554"},"LAPORAN_MODEL_DAN_RENCANA_INTEGRASI.md":{"base64":"IyBMYXBvcmFuIG1vZGVsIEovWiBkYW4gcGVtZWxpaGFyYWFuIG1vZGVsIDI0IGh1cnVmCgojIyAxLiBLZXB1dHVzYW4gYWtoaXIKClNpc3RlbSBtZW5nZ3VuYWthbiAqKmR1YSBtb2RlbCBrbGFzaWZpa2FzaSB5YW5nIGJlcmJlZGEqKiwga2FyZW5hIGplbmlzCm1hc3VrYW5ueWEgYmVyYmVkYS4gQ05OIHlhbmcgc3VkYWggYWRhIG1lbWJhY2EgZ2FtYmFyIHRhbmdhbiwgc2VkYW5na2FuIEdSVQpiYXJ1IG1lbWJhY2EgdXJ1dGFuIHBvc2lzaSB0YW5nYW4gc2VwYW5qYW5nIGdlcmFrYW4uCgp8IEtvbXBvbmVuIHwgRnVuZ3NpIHwgUGVybGFrdWFuIGRhbGFtIG5vdGVib29rIHwKfCAtLS0gfCAtLS0gfCAtLS0gfAp8IENOTiAyNCBodXJ1ZiBzYWF0IGluaSB8IE1lbmdlbmFsaSBodXJ1ZiBzdGF0aXMgc2VsYWluIEovWiB8IERpcGVydGFoYW5rYW4gYnl0ZSBkZW1pIGJ5dGUuIEhhbnlhIGRpZXZhbHVhc2kuIHwKfCBHUlUgSi9aL09USEVSIHwgTWVuZ2VuYWxpIGdlcmFrYW4gSiwgWiwgZGFuIHBvc2UgZGlhbS9nZXJha2FuIGxhaW4geWFuZyBkaXBlbGFqYXJpIHwgRGlsYXRpaCBzZWJhZ2FpIG1vZGVsIGJhcnUuIHwKfCBNZWRpYVBpcGUgSGFuZCBMYW5kbWFya2VyIHwgTWVuZW50dWthbiAyMSBsYW5kbWFyayB0YW5nYW4gfCBNZW1ha2FpIG1vZGVsIHNpYXAgcGFrYWkuIFRpZGFrIGRpbGF0aWggdWxhbmcuIHwKCkNOTiBkYW4gR1JVIHRpZGFrIGRpamFkaWthbiBzYXR1IGZpbGUgbW9kZWwgYXRhdSBkaXR1bXB1ayBib2JvdG55YS4gTmFudGlueWEKa2VkdWFueWEgZGFwYXQgYmVrZXJqYSBkaSBzYXR1IGFwbGlrYXNpIG1lbGFsdWkgcGVuZ2F0dXJhbiBoYXNpbCBwcmVkaWtzaS4KTWVkaWFQaXBlIHNlbmRpcmkganVnYSBtZW5nZ3VuYWthbiBtb2RlbCBNTCBzaWFwIHBha2FpLCB0ZXRhcGkgYnVrYW4gbW9kZWwKa2xhc2lmaWthc2kgaHVydWYgeWFuZyBkaWxhdGloIG9sZWggbm90ZWJvb2sgaW5pLgoKIyMgMi4gQXBhIHlhbmcgZGlsYWt1a2FuIHRlcmhhZGFwIG1vZGVsIGxhbWEKCk1vZGVsIGFzYWwgYWRhbGFoIGBjYW1lcmFfcm9idXN0X3YxYCwgdWt1cmFuICoqMzI2LjEwOCBieXRlIC8gMzE4LDQ2IEtpQioqLgpTSEEtMjU2OgoKYGBgdGV4dAo3NGY1MmFiMGQ2ZjA5MWIzNzYwMGNlNjVjNGI3Nzk3YTEwMmM2ZTI2MTEyYjZjMmU0ODI0NDgzYjE2ODE0MGJjCmBgYAoKTm90ZWJvb2sgbWVueWVydGFrYW4gc2FsaW5hbiBmaWxlIGRhcmkgYXBsaWthc2kgc2VrYXJhbmcuIFNhYXQgYmVyamFsYW4sIG1vZGVsCmRpc2FsaW4ga2VtYmFsaSBrZSBmb2xkZXIgaGFzaWwgZGFuIGhhc2gga2VkdWEgZmlsZSBkaWJhbmRpbmdrYW4uIEJvYm90LCB1cnV0YW4KMjQgbGFiZWwsIG1hc3VrYW4gZ3JheXNjYWxlIDI4IMOXIDI4IGJlcm5pbGFpIDDigJMyNTUsIGRhbiBub3JtYWxpc2FzaSBpbnRlcm5hbAptb2RlbCBkaXBlcnRhaGFua2FuLiBUaWRhayBhZGEgQ05OIGJhcnUsIGZpbmUtdHVuaW5nLCBrdWFudGlzYXNpIHVsYW5nLCBhdGF1CnRyYWluaW5nIG1lbmdndW5ha2FuIHRlc3QgZGF0YSB1bnR1ayBtZW5nZWphciBhbmdrYSB0ZXJ0ZW50dS4KCktvZGUgRmx1dHRlciwgbW9kZWwgZGkgYXBsaWthc2ksIGRhbiBhbWJhbmcgcHJlZGlrc2kgYXBsaWthc2kgdGlkYWsgZGllZGl0IG9sZWgKd29ya2Zsb3cgaW5pLiBBcnRpbnlhLCBla3NwZXJpbWVuIGluaSB0aWRhayBtZW5nZ2FudGkgc2lzdGVtIHByZWRpa3NpIHlhbmcgc2FhdAppbmkgZGlndW5ha2FuLiBVbnR1ayBpbnB1dCBkYW4gcnVudGltZSB5YW5nIHNhbWEsIG1vZGVsIGlkZW50aWsgbWVtcGVydGFoYW5rYW4KZnVuZ3NpIHByZWRpa3NpbnlhLiBQZW5nZ3VuYWFuIGNyb3AgYmVyYmVkYSBwYWRhIGRpYWdub3N0aWsgZGFwYXQgbWVuZ2hhc2lsa2FuCnByZWRpa3NpIGJlcmJlZGEga2FyZW5hIGdhbWJhciBtYXN1a2FubnlhIGJlcmJlZGEsIGJ1a2FuIGthcmVuYSBib2JvdCBiZXJ1YmFoLgoKIyMgMy4gS2VrdXJhbmdhbiB5YW5nIGRpcGVyYmFpa2kKCnwgVGVtdWFuIHNlYmVsdW1ueWEgfCBZYW5nIGRpbGFrdWthbiBzZWthcmFuZyB8IERhbXBhayB0ZXJoYWRhcCBtb2RlbCBsYW1hIHwKfCAtLS0gfCAtLS0gfCAtLS0gfAp8IE5vdGVib29rIGxhbWEgbWVuZWJhayBrb3RhayB0YW5nYW4gZGFyaSB3YXJuYSB8IEV2YWx1YXNpIGZvdG8gbWVtYWthaSBNZWRpYVBpcGUgSU1BR0UsIGNyb3AgcGVyc2VnaSBjb250ZXh0L3RpZ2h0IGRhbiBhdHVyYW4ga2VwdXR1c2FuIGFwbGlrYXNpIHwgTWVtcGVyYmFpa2kgZXZhbHVhc2kvcHJlcHJvY2Vzc2luZywgdGlkYWsgbWVuZ3ViYWggYm9ib3QgfAp8IENhY2hlIGxhbWEgdGlkYWsgbWVtZXJpa3NhIGtlc2VzdWFpYW4ga29uZmlndXJhc2kgfCBDYWNoZSBtZW1ha2FpIGhhc2ggc3VtYmVyLCBrb25maWd1cmFzaSwga29kZSwgdmVyc2kgZGVwZW5kZW5jeSBkYW4gZmlsZSBkZXRla3RvciB8IERhdGEgbGFtYSB0aWRhayBkaWFtLWRpYW0gZGlndW5ha2FuIGRlbmdhbiBrb25maWd1cmFzaSBiZXJiZWRhIHwKfCBFdmFsdWFzaSB0aWdhIGNyb3AgZGloaXR1bmcgc2ViYWdhaSB0aWdhIHNhbXBlbCB8IERpYWdub3N0aWsgbWVtYmVyaSBzYXR1IGtlcHV0dXNhbiBha2hpciBwZXIgZm90bywgdGVybWFzdWsgZ2FnYWwgZGV0ZWtzaS9wZW5vbGFrYW4gfCBEZW5vbWluYXRvciBkYW4gcGVueWViYWIgZ2FnYWwgbGViaWggamVsYXMgfAp8IFRlc3QgZGlwYW50YXUgdGlhcCBlcG9jaCBkYW4gZXBvY2ggdGVyYWtoaXIgZGlzaW1wYW4gfCBQYWRhIHRyYWluaW5nIEovWiwgdmFsaWRhdGlvbiBtZW1pbGloIGNoZWNrcG9pbnQgdGVyYmFpayBkYW4gdGVzdCBkaWphbGFua2FuIHNldGVsYWhueWEgfCBCZXJsYWt1IHBhZGEgdHJhaW5pbmcgbW9kZWwgYmFydS4gVGlkYWsgbWVuZ3ViYWggcml3YXlhdCBtb2RlbCBsYW1hIHwKfCBBa3VyYXNpIHRhbnBhIHBlbmplbGFzYW4ga29uZGlzaSBkYXRhIHwgTGFwb3JhbiBtZW1pc2Foa2FuIE1OSVNULCBkaWFnbm9zdGlrIEhGLCB2aWRlbyBueWF0YSBKL1osIE9USEVSIHNpbnRldGlzLCBjb3ZlcmFnZSwgZGFuIGNvbmZpZGVuY2UgfCBNZW5naGluZGFyaSBrbGFpbSBha3VyYXNpIGthbWVyYSBiZXJkYXNhcmthbiBzYXR1IGFuZ2thIGRhdGFzZXQgfAoKR2VvbWV0cmkgY3JvcCBkYW4gcGVuZ2dhYnVuZ2FuIHNrb3IgbWVuZ2lrdXRpIGFwbGlrYXNpIHlhbmcgZGlwZXJpa3NhOiBjb250ZXh0CnBhZGRpbmcgMSw2MCwgdGlnaHQgMSw0MCwgaGFzaWwgY29udGV4dCBrdWF0IGRhcGF0IGRpdGVyaW1hIGxhbmdzdW5nIHBhZGEgMCw5MCwKc2VsYWluIGl0dSBza29yIGRpZ2FidW5nIDAsNTUvMCw0NS4gS2V0aWRha3NlcGFrYXRhbiBjcm9wIGRpYmF0YXNpIGRpIGJhd2FoCmFtYmFuZyBwZW5lcmltYWFuLiBSZXNpemUgT3BlbkNWIG1lbmRla2F0aSByZXNpemUgYXZlcmFnZSBGbHV0dGVyOyBrZXNhbWFhbgpudW1lcmlrIHNldGlhcCBwaWtzZWwgZGkgcGVyYW5na2F0IGJlbHVtIGRpa2xhaW0gdGVsYWggdGVydWppLgoKTW9kZWwgbGFtYSBwZXJuYWggYmVsYWphciBkYXJpIHN1bWJlciBIdWdnaW5nIEZhY2UuIEtlYW5nZ290YWFuIGNhY2hlIHRyYWluaW5nCmxhbWFueWEgdGlkYWsgdGVyc2VkaWEsIHNlaGluZ2dhIGV2YWx1YXNpIHNlbHVydWggZm90byBIRiBkaWJlcmkgbmFtYQoqKmRpYWdub3N0aWsqKiwgYnVrYW4gdGVzdCBpbmRlcGVuZGVuIGJhcnUuIE5pbGFpIGJhcnUgdGlkYWsgbGFuZ3N1bmcgc2V0YXJhCmRlbmdhbiBhbmdrYSA0NCwyNjQlIHRlcmRhaHVsdSB5YW5nIG1lbWFrYWkgMi40MDAgZm90byBkYW4gdGlnYSBjcm9wIGhldXJpc3Rpay4KCiMjIDQuIENhcmEga2VyamEgbW9kZWwgSiBkYW4gWgoKSHVydWYgSi9aIG1lbWJ1dHVoa2FuIGluZm9ybWFzaSBnZXJha2FuLiBQb3NlIGF3YWwgc2FqYSBkYXBhdCBtZW55ZXJ1cGFpIGh1cnVmCnN0YXRpcy4gS2FyZW5hIGl0dSwgc2F0dSBmb3RvIHRpZGFrIGN1a3VwIHVudHVrIG1lbWFzdGlrYW4gbGludGFzYW4gaHVydWZueWEuCgpBbHVyIHRyYWluaW5nOgoKMS4gVW5kdWggdmlkZW8gSi9aIFNpZ05OIGRhbiBwZXJpa3NhIGhhc2ggdW50dWsgbWVuZ2hhcHVzIGR1cGxpa2FzaSBwZXJzaXMuCjIuIFBpc2Foa2FuIHZpZGVvIGFzbGkgbWVuamFkaSB0cmFpbiwgdmFsaWRhdGlvbiwgZGFuIHRlc3Qgc2ViZWx1bSBtZW1idWF0CiAgIHZhcmlhc2kuIFR1cnVuYW4gc2F0dSB2aWRlbyB0ZXRhcCBiZXJhZGEgcGFkYSBwZW1iYWdpYW4gdmlkZW8gaW5kdWtueWEuCjMuIE1lZGlhUGlwZSBtZW1iYWNhIHZpZGVvIHNlc3VhaSB0aW1lc3RhbXAuIFBvc2lzaSAyMSB0aXRpayB4L3kveiBkYW4gcGVuYW5kYQogICBrZWJlcmFkYWFuIHRhbmdhbiBkaXJla2FtLCBkZW5nYW4gdGFyZ2V0IHNhbXBsaW5nIDE1IG9ic2VydmFzaSBwZXIgZGV0aWsuCjQuIFViYWggc2F0dSB2aWRlbyBtZW5qYWRpIDQ4IGxhbmdrYWggd2FrdHUgZGVuZ2FuIDY0IG5pbGFpIHRpYXAgbGFuZ2thaC4KICAgVGl0aWsgYWN1YW4gZGFuIHNrYWxhIHRldGFwIHVudHVrIHNlbHVydWggdmlkZW8sIHNlaGluZ2dhIGxpbnRhc2FuCiAgIHBlcmdlbGFuZ2FuIHRpZGFrIGhpbGFuZyBha2liYXQgbm9ybWFsaXNhc2kgdGlhcCBmcmFtZS4KNS4gTGF0aWggR1JVNjQgZGVuZ2FuIERlbnNlMzIgZGFuIGtlbHVhcmFuIHRpZ2Ega2VsYXM6IEosIFosIE9USEVSLiBBdWdtZW50YXNpCiAgIGhhbnlhIGRpdGVyYXBrYW4gcGFkYSB0cmFpbmluZy4gQ2hlY2twb2ludCB0ZXJiYWlrIGRpcGlsaWggZGFyaSB2YWxpZGF0aW9uLgo2LiBFdmFsdWFzaSB0ZXN0LCBsYWx1IHNpbXBhbiBtb2RlbCwgYnVrdGksIGRhbiBiYXRhcyBwZW5nZ3VuYWFuLgoKR1JVIG1lbXBlbGFqYXJpIGh1YnVuZ2FuIGFudGFyd2FrdHUuIFBvbGEgZ2VyYWthbiBKIGRhbiBnYXJpcyB6aWd6YWcgWiBkaW5pbGFpCmRhcmkgcmFuZ2thaWFuIHBvc2lzaSwgYnVrYW4gaGFueWEgc2F0dSBiZW50dWsgdGFuZ2FuLiBWaWRlbyB5YW5nIGRldGVrc2kKdGFuZ2FubnlhIHRlcmxhbHUgc2VkaWtpdCBhdGF1IGhpbGFuZyB0ZXJsYWx1IGxhbWEgZGl0b2xhayBkYW4gZGljYXRhdC4KCk9USEVSIGRpcGVybHVrYW4gYWdhciBtb2RlbCB0aWRhayBtZW1ha3NhIHNlbXVhIG1hc3VrYW4gbWVuamFkaSBKIGF0YXUgWi4KUnVuIGRlZmF1bHQgbWVtYWthaSBjb250b2ggcG9zZSBkaWFtIHNpbnRldGlzLiBJbmkgYmVsdW0gY3VrdXAgbWVtYnVrdGlrYW4Ka2V0YWhhbmFuIHRlcmhhZGFwIHNlbXVhIGdlcmFrYW4gbWFudXNpYS4gUmVrYW1hbiBnZXJha2FuIGJpYXNhIGRhcmkgcGVuZ2d1bmEKYmVyYmVkYSBkYXBhdCBkaXRhbWJhaGthbiBtZWxhbHVpIGZvbGRlciBgbmVnYXRpdmVfdmlkZW9zL2AuIExhcG9yYW4gbWVtaXNhaGthbgpjb250b2ggc2ludGV0aXMgZGFuIHJla2FtYW4gYXNsaS4KClRhcmdldCA5OSw1JSBkaWNhdGF0IGFwYSBhZGFueWEuIFRyYWluaW5nIHRpZGFrIGRpdWxhbmcgYmVyZGFzYXJrYW4gdGVzdCBzYW1wYWkKdGVybGloYXQgc2VtcHVybmEuIFNlZWQgdHJhaW5pbmcgYXdhbCA0Miwgc2VkYW5na2FuIHNwbGl0X3NlZWQgMjAyNiBtZW5qYWdhCnBlbWJhZ2lhbiBzdW1iZXIuIE5vbW9yIHNlZWQgYnVrYW4gdWt1cmFuIGtlcGludGFyYW4gbW9kZWwuCgojIyA1LiBSZW5jYW5hIHBlbmVyYXBhbiBkaSBGbHV0dGVyIHNldGVsYWggaGFzaWwgdHJhaW5pbmcgYWRhCgpCZWx1bSBhZGEgaW1wbGVtZW50YXNpIGF0YXUgcGVydWJhaGFuIGFwbGlrYXNpIHBhZGEgdGFoYXAgaW5pLiBSZW5jYW5hbnlhOgoKYGBgdGV4dApLYW1lcmEgLT4gTWVkaWFQaXBlIC0+IGxhbmRtYXJrIGRhbiBhcmVhIHRhbmdhbgogICAgICAgICAgICAgICAgICAgICAgICAgfAogICAgICAgICAgICAgICAgICAgICAgICAgKy0+IGdhbWJhciBjcm9wIC0+IENOTiAyNCBodXJ1ZiB5YW5nIHN1ZGFoIGFkYQogICAgICAgICAgICAgICAgICAgICAgICAgfAogICAgICAgICAgICAgICAgICAgICAgICAgKy0+IHVydXRhbiBsYW5kbWFyayAtPiBHUlUgSi9aL09USEVSCgpLZWR1YSBoYXNpbCAtPiBhdHVyYW4gcGVuZXJpbWFhbiBkYW4gcGVuZ3VsYW5nYW4gLT4gc2F0dSByYW5na2FpYW4gaHVydWYKYGBgCgpNb2RlbCBDTk4gdGV0YXAgbWVuZXJpbWEgZ2FtYmFyIHNlcGVydGkgc2VrYXJhbmcuIE1vZGVsIEdSVSBtZW5lcmltYSBidWZmZXIKbGFuZG1hcmsgYmVydGltZXN0YW1wLiBOb3JtYWxpc2FzaSwgdXJ1dGFuIGZpdHVyLCBzYW1wbGluZywgZGFuIHBhbmphbmcgYnVmZmVyCmhhcnVzIG1lbmdpa3V0aSBgcHJlcHJvY2Vzc2luZ19jb250cmFjdC5qc29uYCBoYXNpbCB0cmFpbmluZy4KCkRldGVrdG9yIGFwbGlrYXNpIHNla2FyYW5nIGRpamFkd2Fsa2FuIHNla2l0YXIgMzIwIG1zLiBSYW5na2FpYW4gZ2VyYWthbiBwZXJsdQpvYnNlcnZhc2kgbGViaWggcmFwYXQgZGFuIGJlbmFyLWJlbmFyIGJhcnUuIEphZHdhbCBNZWRpYVBpcGUgcGVybHUgZGlzZXN1YWlrYW4KZGFuIGRpdWt1ciBkaSBWaXZvLCB0YW5wYSBtZW55YWxpbiBsYW5kbWFyayBsYW1hIHNlb2xhaCBtZW5qYWRpIGZyYW1lIGJhcnUuCkNOTiB0aWRhayBoYXJ1cyBkaWphbGFua2FuIHBhZGEgc2V0aWFwIG9ic2VydmFzaSBsYW5kbWFyay4KCkF0dXJhbiBwZW5nZ2FidW5nYW4gaGFydXMgZGl1amkgYWdhciBwb3NlIGF3YWwgSiB0aWRhayB0ZXJjYXRhdCBzZWJhZ2FpIEkKdGFtYmFoYW4sIHNhdHUgZ2VyYWthbiB0aWRhayBtZW5naGFzaWxrYW4gaHVydWYgYmVydWxhbmcsIGRhbiBnZXJha2FuIHJhZ3UKdGlkYWsgZGlwYWtzYWthbiBtZW5qYWRpIEovWi4gUHJlZGlrc2kgZGl0ZXJpbWEgc2V0ZWxhaCBidWt0aSBnZXJha2FuIGN1a3VwLgpKaWthIGJlbHVtIGplbGFzLCBhcGxpa2FzaSBtZW51bmdndSBhdGF1IG1lbm9sYWsuIEZvdG8vdXBsb2FkIHR1bmdnYWwgdGV0YXAKbWVuaWxhaSAyNCBodXJ1ZiBzdGF0aXM7IEovWiBtZW1lcmx1a2FuIHZpZGVvIGF0YXUgdXJ1dGFuIGZyYW1lLgoKU2V0ZWxhaCBpbXBsZW1lbnRhc2kgbmFudGksIGphbGFua2FuIHVqaSByZWdyZXNpIDI0IGh1cnVmIHlhbmcgc2FtYSBzZWJlbHVtCmRhbiBzZXN1ZGFoIHBlcnViYWhhbi4gQmFuZGluZ2thbiBha3VyYXNpLCBwZW5vbGFrYW4sIHdha3R1IHJlc3BvbnMsIGRhbgpwZW5ndWxhbmdhbiBodXJ1ZiBwYWRhIHBlcmFuZ2thdCB5YW5nIHNhbWEuIFRhbWJhaGthbiB1amkgSi9aLCBwZXJwaW5kYWhhbgpwb3NlLCB0YW5nYW4gZGlhbSwgdGFuZ2FuIGhpbGFuZywgc2VydGEgZ2VyYWthbiBsYWluLiBBa3RpZmthbiBmaXR1ciBKL1oKc2V0ZWxhaCBoYXNpbCBnYWJ1bmdhbiB0ZXJidWt0aSBtZW1hZGFpLiBIYXNoIG1vZGVsIGxhbWEgc2FqYSB0aWRhayBtZW5qYW1pbgpiYWh3YSBhdHVyYW4gcGVuZ2dhYnVuZ2FuIGJhcnUgdGlkYWsgbWVtaWxpa2kgYnVnLCBzZWhpbmdnYSB1amkgYXBsaWthc2kKdGV0YXAgZGlwZXJsdWthbiBwYWRhIHRhaGFwIGludGVncmFzaSB0ZXJzZWJ1dC4KCiMjIDYuIEJ1a3RpIHlhbmcgYWthbiBkaXBlcm9sZWgKCk1vZGVsIHN0YXRpcyBtZW1pbGlraSBsYXBvcmFuIGRpYWdub3N0aWsgZGFuIGlkZW50aXRhcyBmaWxlLiBNb2RlbCBKL1ogbWVtaWxpa2kKa3VydmEgdHJhaW4vdmFsaWRhdGlvbiwgQ1NWIHBlciBlcG9jaCwgY2hlY2twb2ludCB0ZXJiYWlrLCBjb25mdXNpb24gbWF0cml4LApwcmVjaXNpb24vcmVjYWxsL0YxLCBwcmVkaWtzaSB0aWFwIHZpZGVvLCBjb3ZlcmFnZSBkZXRla3NpLCBkYW4gY2F0YXRhbiBrZXNhbGFoYW4uClRGTGl0ZSBmbG9hdDMyL2Zsb2F0MTYgZGlzZXJ0YWkgdWt1cmFuIGFrdHVhbCwgaGFzaCwga29udHJhayBpbnB1dC9vdXRwdXQsIGRhbgpwYXJpdHkgcGFkYSB2YWxpZGF0aW9uLiBUZXN0IEdSVSBtZW5nZ3VuYWthbiBjaGVja3BvaW50IEtlcmFzIHRlcmJhaWssIGJ1a2FuCnBlbmd1amlhbiBmdWxsLXRlc3QgVEZMaXRlIHBhZGEgSFAuCgpMYW5kbWFyayBtZW50YWggYmVydGltZXN0YW1wLCBtYW5pZmVzdCBwZW1iYWdpYW4gZGF0YSwga29uZmlndXJhc2ksIHNvdXJjZSBjb2RlLAp2ZXJzaSBkZXBlbmRlbmN5LCBkYW4gbW9kZWwgY2FyZCBpa3V0IHNhdHUgWklQIGhhc2lsLiBVa3VyYW4gbW9kZWwgYmFydSBzZXJ0YQphbmdrYSBha3VyYXNpIGFrdHVhbCBiYXJ1IGRpa2V0YWh1aSBzZXRlbGFoIG5vdGVib29rIGRpamFsYW5rYW4gZGkgQ29sYWIuCgpUaWRhayBhZGEga2xhaW0gYmFod2EgdmlkZW8gU2lnTk4gbWVtYnVrdGlrYW4ga2VtYW1wdWFuIHBhZGEgcGVuZ2d1bmEgYmFydToKaWRlbnRpdGFzIHBlcmFnYSBkYXRhc2V0IGJlbHVtIHRlcnZlcmlmaWthc2kuIFRpZGFrIGFkYSBrbGFpbSBiYWh3YSBPVEhFUgpzaW50ZXRpcyBtZW1idWt0aWthbiBrZW1hbXB1YW4gbWVub2xhayBzZW11YSBnZXJha2FuIGxhaW4uIExhcG9yYW4gbWVtYmVyaWthbgpzdGF0dXMgcGVuZWxpdGlhbiB5YW5nIHNlc3VhaSBidWt0aSwgYnVrYW4gamFtaW5hbiAxMDAlIHBhZGEga2FtZXJhIG55YXRhLgo=","sha256":"69c59495487302de94b0470597fbf2722c9e9530c6587d6fa6526588d1b11736"},"baseline_model/model_float16.tflite":{"base64":"...","sha256":"74f52ab0d6f091b37600ce65c4b7797a102c6e26112b6c2e4824483b168140bc"},"baseline_model/labels.json":{"base64":"ewogICJzY2hlbWFfdmVyc2lvbiI6IDEsCiAgImxhYmVscyI6IFsKICAgICJBIiwKICAgICJCIiwKICAgICJDIiwKICAgICJEIiwKICAgICJFIiwKICAgICJGIiwKICAgICJHIiwKICAgICJIIiwKICAgICJJIiwKICAgICJLIiwKICAgICJMIiwKICAgICJNIiwKICAgICJOIiwKICAgICJPIiwKICAgICJQIiwKICAgICJRIiwKICAgICJSIiwKICAgICJTIiwKICAgICJUIiwKICAgICJVIiwKICAgICJWIiwKICAgICJXIiwKICAgICJYIiwKICAgICJZIgogIF0sCiAgIm91dHB1dF9pbmRleF90b19sYWJlbCI6IHsKICAgICIwIjogIkEiLAogICAgIjEiOiAiQiIsCiAgICAiMiI6ICJDIiwKICAgICIzIjogIkQiLAogICAgIjQiOiAiRSIsCiAgICAiNSI6ICJGIiwKICAgICI2IjogIkciLAogICAgIjciOiAiSCIsCiAgICAiOCI6ICJJIiwKICAgICI5IjogIksiLAogICAgIjEwIjogIkwiLAogICAgIjExIjogIk0iLAogICAgIjEyIjogIk4iLAogICAgIjEzIjogIk8iLAogICAgIjE0IjogIlAiLAogICAgIjE1IjogIlEiLAogICAgIjE2IjogIlIiLAogICAgIjE3IjogIlMiLAogICAgIjE4IjogIlQiLAogICAgIjE5IjogIlUiLAogICAgIjIwIjogIlYiLAogICAgIjIxIjogIlciLAogICAgIjIyIjogIlgiLAogICAgIjIzIjogIlkiCiAgfSwKICAidW5zdXBwb3J0ZWRfZHluYW1pY19sZXR0ZXJzIjogWwogICAgIkoiLAogICAgIloiCiAgXQp9Cg==","sha256":"a97b4c4f108d2c7e6f7e6f9bff7ee316a990808ff4f64bb1f1120a0ebbf61588"},"baseline_model/model_metadata.json":{"base64":"ew0KICAic2NoZW1hX3ZlcnNpb24iOiAxLA0KICAibW9kZWxfbmFtZSI6ICJMYXRpaElzeWFyYXQgLSBjYW1lcmEtZG9tYWluIHJvYnVzdCIsDQogICJtb2RlbF92ZXJzaW9uIjogImNhbWVyYV9yb2J1c3RfdjEiLA0KICAiaW5wdXRfY29udHJhY3QiOiB7DQogICAgInNoYXBlIjogWw0KICAgICAgMSwNCiAgICAgIDI4LA0KICAgICAgMjgsDQogICAgICAxDQogICAgXSwNCiAgICAiZHR5cGUiOiAiZmxvYXQzMiIsDQogICAgInZhbHVlX3JhbmdlX2JlZm9yZV9tb2RlbCI6IFsNCiAgICAgIDAuMCwNCiAgICAgIDI1NS4wDQogICAgXSwNCiAgICAiY29sb3Jfc3BhY2UiOiAiZ3JheXNjYWxlIiwNCiAgICAibm9ybWFsaXphdGlvbiI6ICJSZXNjYWxpbmcoMS8yNTUpIGluc2lkZSB0aGUgbW9kZWwiLA0KICAgICJtb2JpbGVfbXVzdF9kaXZpZGVfYnlfMjU1IjogZmFsc2UNCiAgfSwNCiAgIm91dHB1dF9jb250cmFjdCI6IHsNCiAgICAic2hhcGUiOiBbDQogICAgICAxLA0KICAgICAgMjQNCiAgICBdLA0KICAgICJkdHlwZSI6ICJmbG9hdDMyIiwNCiAgICAibGFiZWxzIjogWw0KICAgICAgIkEiLA0KICAgICAgIkIiLA0KICAgICAgIkMiLA0KICAgICAgIkQiLA0KICAgICAgIkUiLA0KICAgICAgIkYiLA0KICAgICAgIkciLA0KICAgICAgIkgiLA0KICAgICAgIkkiLA0KICAgICAgIksiLA0KICAgICAgIkwiLA0KICAgICAgIk0iLA0KICAgICAgIk4iLA0KICAgICAgIk8iLA0KICAgICAgIlAiLA0KICAgICAgIlEiLA0KICAgICAgIlIiLA0KICAgICAgIlMiLA0KICAgICAgIlQiLA0KICAgICAgIlUiLA0KICAgICAgIlYiLA0KICAgICAgIlciLA0KICAgICAgIlgiLA0KICAgICAgIlkiDQogICAgXQ0KICB9LA0KICAidHJhaW5pbmdfZGF0YSI6IHsNCiAgICAic2lnbl9tbmlzdF9yb3dzIjogMjc0NTUsDQogICAgImF1Z21lbnRlZF9zaWduX21uaXN0X3Jvd3MiOiAxMDk4MjAsDQogICAgInJlYWxfcGhvdG9zX3RyYWluIjogNzYzMywNCiAgICAicmVhbF9waG90b3NfdGVzdCI6IDI0MDAsDQogICAgInRvdGFsX3Jvd3MiOiAxMzI3MTksDQogICAgImF1Z21lbnRhdGlvbiI6ICJwb2xhcml0eSBmbGlwLCA1NS0xMDAlIGhhbmQgc2NhbGUsIHJhbmRvbSBiYWNrZ3JvdW5kIHdpdGggZ3VhcmFudGVlZCBjb250cmFzdCwgcm90YXRpb24vc2hlYXIsIGdhbW1hLCBibHVyLCBub2lzZSINCiAgfSwNCiAgIm1ldHJpY3MiOiB7DQogICAgImNsZWFuIjogew0KICAgICAgImFjY3VyYWN5IjogMC45OTk4NjA1Njg4Nzg5NzM4LA0KICAgICAgIm1lYW5fY29uZmlkZW5jZSI6IDAuOTk4NTkxOTU5NDc2NDcxDQogICAgfSwNCiAgICAiY2FtZXJhX3NpbSI6IHsNCiAgICAgICJhY2N1cmFjeSI6IDAuODY3Njc5ODY2MTQ2MTIzOCwNCiAgICAgICJtZWFuX2NvbmZpZGVuY2UiOiAwLjg4NzY3NTA0NjkyMDc3NjQNCiAgICB9LA0KICAgICJyZWFsX3Bob3RvcyI6IHsNCiAgICAgICJhY2N1cmFjeSI6IDAuNDQyNjM4ODg4ODg4ODg4OSwNCiAgICAgICJtZWFuX2NvbmZpZGVuY2UiOiAwLjQ1NDczMzg0ODU3MTc3NzM0DQogICAgfQ0KICB9LA0KICAidGZsaXRlIjogew0KICAgICJwYXRoIjogIm1vZGVsX2Zsb2F0MTYudGZsaXRlIiwNCiAgICAicXVhbnRpemF0aW9uIjogImZsb2F0MTZfd2VpZ2h0cyIsDQogICAgInNpemVfYnl0ZXMiOiAzMjYxMDgsDQogICAgInNoYTI1NiI6ICI3NGY1MmFiMGQ2ZjA5MWIzNzYwMGNlNjVjNGI3Nzk3YTEwMmM2ZTI2MTEyYjZjMmU0ODI0NDgzYjE2ODE0MGJjIiwNCiAgICAia2VyYXNfdGZsaXRlX2xhYmVsX2FncmVlbWVudCI6IDEuMCwNCiAgICAibWF4aW11bV9hYnNvbHV0ZV9zY29yZV9kaWZmZXJlbmNlIjogMC4wMDI3ODYyNDg5MjIzNDgwMjI1DQogIH0sDQogICJjYW1lcmFfd2FybmluZyI6ICJBbmdrYSByZWFsX3Bob3RvcyBhZGFsYWggZm90byB3ZWJjYW0sIGJ1a2FuIGthbWVyYSBIUC4gVWt1ciB1bGFuZyBkaSBIUCBzZWJlbHVtIG1lbnlpbXB1bGthbiBha3VyYXNpLiINCn0NCg==","sha256":"5babc2c0129b53dd8cd88b2448c00d39564460e8ce01d09151b4d1d5e16718f9"}}' )
for name, item in EMBEDDED_FILES.items():
    raw = base64.b64decode(item['base64'])
    assert hashlib.sha256(raw).hexdigest() == item['sha256'], name
    target = SOURCE_DIR / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(raw)
print('Model 24 huruf:', len((SOURCE_DIR / 'baseline_model/model_float16.tflite').read_bytes()), 'byte, bobot tetap.')


In [ ]:
# Konfigurasi utama. Tidak perlu membesarkan seed untuk menaikkan akurasi.
CONFIG = {
    'seed': 42,
    'split_seed': 2026,
    'static_mode': 'evaluate_existing',
    'dynamic_epochs': 60,
    'batch_size': 256,
    'learning_rate': 0.001,
    'sequence_length': 48,
    'dynamic_sample_fps': 15,
    'validation_fraction': 0.15,
    'test_fraction': 0.15,
    'min_confidence': 0.60,
    'target_dynamic_accuracy': 0.995,
    'require_gpu': True,
    'use_drive': True,
    'drive_dir': '/content/drive/MyDrive/LatihIsyarat_Training_J_Z',
    'auto_download': False,  # Unduhan dipicu notebook setelah proses selesai.
}

from google.colab import drive
drive.mount('/content/drive')
print('Drive:', CONFIG['drive_dir'])
print('Opsional: rekaman OTHER asli di negative_videos/orang_01/, orang_02/, orang_03/.')


In [ ]:
# Jalankan evaluasi + training. Log ditampilkan langsung, satu ZIP diunduh di akhir.
import os
import subprocess
import sys
from google.colab import files

pointer = SOURCE_DIR / 'runtime_result.json'
pointer.unlink(missing_ok=True)
CONFIG['result_pointer'] = str(pointer)
configuration = SOURCE_DIR / 'runtime_config.json'
configuration.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
environment = {**os.environ, 'PYTHONUNBUFFERED': '1', 'TF_CPP_MIN_LOG_LEVEL': '2'}
process = subprocess.Popen(
    [sys.executable, str(SOURCE_DIR / 'train_all.py'), '--config', str(configuration)],
    cwd=SOURCE_DIR, env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='')
exit_code = process.wait()
if pointer.exists():
    RESULT = json.loads(pointer.read_text(encoding='utf-8'))
    print('\nSTATUS:', RESULT['status'])
    print('FOLDER:', RESULT['run_directory'])
    print('ZIP:', RESULT['zip_path'])
    try:
        files.download(RESULT['zip_path'])
    except Exception as error:
        print('Unduh ZIP melalui Drive jika browser menahan unduhan:', error)
else:
    raise RuntimeError('Proses berhenti sebelum paket hasil dibuat. Lihat pesan error terakhir di atas.')
if exit_code != 0:
    raise RuntimeError('Training belum selesai. ZIP berisi bukti sementara dan traceback, bukan hasil final.')


## Membaca hasil

Ekstrak ZIP menjadi satu folder. Mulai dari `RINGKASAN_HASIL.json`.

- **Static24:** model lama tetap, laporan evaluasi/crop/confusion matrix per huruf.
- **Dynamic J/Z:** checkpoint terbaik, TFLite, ukuran, kurva train/validation, prediksi
  test per video, confusion matrix, landmark bertimestamp, dan kontrak fitur.
- **OTHER sintetis** dan video nyata dilaporkan terpisah. Hasil tinggi pada pose
  diam sintetis tidak membuktikan semua gerakan lain bisa ditolak.
- Manifest split, hash sumber, versi paket, konfigurasi, dan snapshot kode ikut ZIP.
- File dataset mentah tetap di Drive, tidak dibundel ulang ke ZIP bukti.

Setelah hasil diperiksa, barulah tentukan implementasi J/Z pada aplikasi.
